# ADHD Explainable Multimodal AI — Complete & Fixed

## Datasets required
| # | Dataset | Link |
|---|---------|------|
| 1 | HYPERAKTIV | [kaggle.com/datasets/kishore00afk/hyperaktiv](https://www.kaggle.com/datasets/kishore00afk/hyperaktiv) |
| 2 | DETEC-ADHD EEG | [kaggle.com/datasets/danizo/eeg-dataset-for-adhd](https://www.kaggle.com/datasets/danizo/eeg-dataset-for-adhd) |
| 3 | ADHD-200 | [kaggle.com/datasets/kishore00afk/adhd-200-preprocessed](https://www.kaggle.com/datasets/kishore00afk/adhd-200-preprocessed) |


In [5]:
# =============================================================================
# CELL 0 — INSTALL
# =============================================================================
import subprocess
subprocess.run(['pip','install','shap','scipy','scikit-learn','torch',
                'xgboost','matplotlib','seaborn','pandas','openpyxl',
                'nilearn','-q'], check=False)


CompletedProcess(args=['pip', 'install', 'shap', 'scipy', 'scikit-learn', 'torch', 'xgboost', 'matplotlib', 'seaborn', 'pandas', 'openpyxl', 'nilearn', '-q'], returncode=0)

In [6]:
# =============================================================================
# CELL 1 — IMPORTS, SEED, PATHS, UTILITIES
# =============================================================================
import random, os, re, glob, json, warnings
from copy import deepcopy
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

from scipy.signal import welch as sp_welch
from scipy.stats import kurtosis, skew

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import (Dataset, DataLoader, TensorDataset,
                               WeightedRandomSampler)

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              precision_recall_curve, average_precision_score,
                              roc_curve)
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

warnings.filterwarnings('ignore')

# ── Seed ──────────────────────────────────────────────────────────────────
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ── Dataset Paths ─────────────────────────────────────────────────────────
PATHS = {
    'hyperaktiv': '/kaggle/input/datasets/kishore00afk/hyperaktiv',
    'eeg':        '/kaggle/input/datasets/danizo/eeg-dataset-for-adhd',
    'adhd200':    '/kaggle/input/datasets/kishore00afk/adhd-200-preprocessed',
}

# ── XGBoost hardware config ───────────────────────────────────────────────
xgb_v  = tuple(int(x) for x in xgb.__version__.split('.')[:2])
xgb_hw = ({'tree_method': 'hist', 'device': 'cuda'}
           if (torch.cuda.is_available() and xgb_v >= (2, 0))
           else {'tree_method': 'hist'})

# ── Shared utilities ──────────────────────────────────────────────────────
def get_class_weights(y, dev):
    cw = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return torch.FloatTensor(cw).to(dev)

def make_weighted_sampler(y):
    counts  = np.bincount(y)
    weights = 1.0 / counts[y]
    return WeightedRandomSampler(torch.FloatTensor(weights), len(weights))

class EarlyStopping:
    def __init__(self, patience=12, mode='max'):
        self.patience = patience; self.mode = mode
        self.best = None; self.bad = 0; self.best_state = None

    def step(self, val, model):
        improved = (self.best is None
                    or (self.mode == 'max' and val > self.best)
                    or (self.mode == 'min' and val < self.best))
        if improved:
            self.best = val; self.bad = 0
            self.best_state = deepcopy(model.state_dict())
        else:
            self.bad += 1
        return self.bad >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)

def train_nn(model, loader, X_val, y_val, criterion, optimizer,
             scheduler, max_epochs=150, patience=15,
             verbose_every=20, tag='Model'):
    es = EarlyStopping(patience=patience, mode='max')
    for ep in range(max_epochs):
        model.train(); total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out    = model(xb)
            logits = out[0] if isinstance(out, tuple) else out
            loss   = criterion(logits, yb)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        model.eval()
        with torch.no_grad():
            out_v  = model(X_val)
            lv     = out_v[0] if isinstance(out_v, tuple) else out_v
            pv     = lv.argmax(1).cpu().numpy()
            f1v    = f1_score(y_val.cpu().numpy(), pv, zero_division=0)
            try:    scheduler.step(f1v)
            except: scheduler.step()
        if (ep + 1) % verbose_every == 0:
            print(f"  [{tag}] Ep {ep+1:3d}  "
                  f"loss={total_loss/len(loader):.4f}  val_f1={f1v:.4f}")
        if es.step(f1v, model):
            print(f"  [{tag}] Early stop ep {ep+1}  best_f1={es.best:.4f}")
            break
    es.restore(model)
    return model

print("Utilities ready.")


Device: cuda
Utilities ready.


In [7]:
# =============================================================================
# CELL 2 — CLINICAL BRANCH: Load HYPERAKTIV + Build Feature Matrix
# =============================================================================
print("\n" + "="*65)
print("CLINICAL BRANCH — HYPERAKTIV")
print("="*65)
print("Dataset: kaggle.com/datasets/kishore00afk/hyperaktiv")
print("  51 subjects (25 ADHD, 26 HC) | Clinical + CPT + actigraphy + HRV")
print()

HP = PATHS['hyperaktiv']

patient_info = pd.read_csv(os.path.join(HP, 'patient_info.csv'), sep=';')
features_df  = pd.read_csv(os.path.join(HP, 'features.csv'),     sep=';')

cpt_path = os.path.join(HP, 'CPT_II_ConnersContinuousPerformanceTest.csv')
cpt_df   = pd.read_csv(cpt_path, sep=';') if os.path.exists(cpt_path) else None

for df in [patient_info, features_df]:
    df['ID'] = df['ID'].astype(int)

print(f"  patient_info: {patient_info.shape}  features: {features_df.shape}")
print(f"  ADHD dist: {patient_info['ADHD'].value_counts().to_dict()}")

# ── Merge features with labels ────────────────────────────────────────────
merged = pd.merge(features_df,
                  patient_info[['ID','ADHD','AGE','SEX']],
                  on='ID', how='inner')

# ── CPT-II features ───────────────────────────────────────────────────────
if cpt_df is not None:
    cpt_df['ID'] = cpt_df['ID'].astype(int)
    score_cols   = [c for c in cpt_df.columns
                    if any(k in c for k in ['TScore','Score','Percent',
                                             'Index','Confidence'])]
    trial_cols   = [c for c in cpt_df.columns if c.startswith('Trial')]
    if trial_cols:
        rt = cpt_df[trial_cols].apply(pd.to_numeric, errors='coerce')
        cpt_feats = pd.DataFrame({'ID': cpt_df['ID']})
        cpt_feats['cpt_rt_mean'] = rt.mean(axis=1)
        cpt_feats['cpt_rt_std']  = rt.std(axis=1)
        cpt_feats['cpt_rt_cv']   = (rt.std(axis=1)
                                    / (rt.mean(axis=1) + 1e-8))
        n = len(trial_cols)
        if n >= 6:
            t = n // 3
            cpt_feats['cpt_vigilance_decrement'] = (
                rt.iloc[:, -t:].mean(axis=1)
                - rt.iloc[:, :t].mean(axis=1))
        if score_cols:
            cpt_feats = pd.concat([cpt_feats, cpt_df[score_cols]], axis=1)
        merged = pd.merge(merged, cpt_feats, on='ID', how='left')

# ── Activity + HRV time-series stats ─────────────────────────────────────
def ts_stats(path, prefix):
    feats = {}
    if not os.path.exists(path):
        return feats
    for fn in sorted(os.listdir(path)):
        if not fn.endswith('.csv'):
            continue
        m = re.search(r'(\d+)', fn)
        if not m:
            continue
        pid = int(m.group(1))
        try:
            df_ts = None
            for sep in [';', ',']:
                try:
                    df_ts = pd.read_csv(os.path.join(path, fn),
                                        sep=sep, skiprows=2)
                    if df_ts.shape[1] > 1:
                        break
                except Exception:
                    pass
            if df_ts is None:
                continue
            num = df_ts.select_dtypes(include=[np.number]).columns
            if len(num) == 0:
                continue
            sig = df_ts[num[-1]].dropna().values
            if len(sig) < 10:
                continue
            feats[pid] = {
                f'{prefix}mean':     float(np.mean(sig)),
                f'{prefix}std':      float(np.std(sig)),
                f'{prefix}median':   float(np.median(sig)),
                f'{prefix}range':    float(np.ptp(sig)),
                f'{prefix}skew':     float(skew(sig)),
                f'{prefix}kurtosis': float(kurtosis(sig)),
            }
            if 'hr' in prefix:
                feats[pid][f'{prefix}rmssd'] = float(
                    np.sqrt(np.mean(np.diff(sig)**2)))
        except Exception:
            continue
    return feats

for feat_dict, prefix in [
    (ts_stats(os.path.join(HP,'activity_data'), 'act_'), 'act_'),
    (ts_stats(os.path.join(HP,'hrv_data'),      'hr_'),  'hr_'),
]:
    if feat_dict:
        df_f = pd.DataFrame.from_dict(feat_dict, orient='index').reset_index()
        df_f.rename(columns={'index': 'ID'}, inplace=True)
        df_f['ID'] = df_f['ID'].astype(int)
        merged = pd.merge(merged, df_f, on='ID', how='left')
        print(f"  Added {prefix.strip('_')} features: {len(feat_dict)} subjects")

# ── Build feature matrix ──────────────────────────────────────────────────
exclude = {'ID','ADHD','AGE','SEX','EDUCATION','MEDICATION','study_id'}
feat_cols = [c for c in merged.columns
             if c not in exclude
             and merged[c].dtype in ['float64','int64','float32','int32']
             and merged[c].nunique() > 1]

X_clin_raw = merged[feat_cols].values.astype(float)
y_clinical  = merged['ADHD'].values.astype(int)
subj_ids    = merged['ID'].values

has_med          = 'MEDICATION' in merged.columns
medication_status = merged['MEDICATION'].values if has_med else None

X_clin_imp = SimpleImputer(strategy='median').fit_transform(X_clin_raw)
X_clin_imp = np.nan_to_num(X_clin_imp, nan=0, posinf=0, neginf=0)

k_feats = min(30, X_clin_imp.shape[1])
selector = SelectKBest(f_classif, k=k_feats)
X_clin_sel = selector.fit_transform(X_clin_imp, y_clinical)
selected_feature_names = [feat_cols[i]
                           for i in selector.get_support(indices=True)]

print(f"\n  Clinical matrix : {X_clin_sel.shape}  "
      f"ADHD={sum(y_clinical==1)}  HC={sum(y_clinical==0)}")
print(f"  Top 5 features  : {selected_feature_names[:5]}")



CLINICAL BRANCH — HYPERAKTIV
Dataset: kaggle.com/datasets/kishore00afk/hyperaktiv
  51 subjects (25 ADHD, 26 HC) | Clinical + CPT + actigraphy + HRV

  patient_info: (103, 33)  features: (116, 788)
  ADHD dist: {0: 52, 1: 51}
  Added act features: 85 subjects
  Added hr features: 80 subjects

  Clinical matrix : (85, 30)  ADHD=45  HC=40
  Top 5 features  : ['ACC__fft_coefficient__attr_"real"__coeff_39', 'ACC__fft_coefficient__attr_"real"__coeff_53', 'ACC__fft_coefficient__attr_"real"__coeff_56', 'ACC__fft_coefficient__attr_"real"__coeff_57', 'ACC__fft_coefficient__attr_"imag"__coeff_97']


In [8]:
# =============================================================================
# CELL 3 — CLINICAL ML MODELS (XGBoost + RF, 5-fold CV)
# =============================================================================
print("\n" + "="*65)
print("CLINICAL MODELS — 5-fold CV")
print("="*65)

models_dict = {
    'XGBoost': xgb.XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, eval_metric='logloss', **xgb_hw),
    'RandomForest': RandomForestClassifier(
        n_estimators=500, max_depth=8, min_samples_leaf=2,
        class_weight='balanced', random_state=42, n_jobs=-1),
}

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"\n  {'Model':<14} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print("  " + "─"*40)
best_clin_auc, best_clin_name, best_clin_model = 0, '', None

for name, mdl in models_dict.items():
    accs, f1s, aucs = [], [], []
    for tri, vai in cv5.split(X_clin_sel, y_clinical):
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X_clin_sel[tri])
        Xva = sc.transform(X_clin_sel[vai])
        m   = deepcopy(mdl); m.fit(Xtr, y_clinical[tri])
        yp  = m.predict(Xva)
        ypr = m.predict_proba(Xva)[:, 1]
        accs.append(accuracy_score(y_clinical[vai], yp))
        f1s.append(f1_score(y_clinical[vai], yp, zero_division=0))
        try:    aucs.append(roc_auc_score(y_clinical[vai], ypr))
        except: aucs.append(0.5)
    ma, mf, mu = np.mean(accs), np.mean(f1s), np.mean(aucs)
    print(f"  {name:<14} {ma:>8.4f} {mf:>8.4f} {mu:>8.4f}")
    if mu > best_clin_auc:
        best_clin_auc = mu; best_clin_name = name
        best_clin_model = deepcopy(mdl)

sc_clin_final = StandardScaler()
X_clin_final  = sc_clin_final.fit_transform(X_clin_sel)
best_clin_model.fit(X_clin_final, y_clinical)
print(f"\n  Best: {best_clin_name}  AUC={best_clin_auc:.4f}")

# ── SHAP — Clinical ───────────────────────────────────────────────────────
print("\n  Computing SHAP for clinical expert...")
explainer_clin   = shap.TreeExplainer(best_clin_model)
shap_values_clin = explainer_clin.shap_values(X_clin_final)

shap_vals_clin_pos = (shap_values_clin[1]
                      if isinstance(shap_values_clin, list)
                         and len(shap_values_clin) > 1
                      else shap_values_clin[0]
                      if isinstance(shap_values_clin, list)
                      else shap_values_clin)

shap_importance_clin = np.abs(shap_vals_clin_pos).mean(axis=0)
top_clin_idx  = np.argsort(shap_importance_clin)[::-1][:10]
top_clin_feat = [selected_feature_names[i] for i in top_clin_idx]
top_clin_vals = shap_importance_clin[top_clin_idx]
print(f"  Top clinical SHAP features: {top_clin_feat[:5]}")



CLINICAL MODELS — 5-fold CV

  Model               Acc       F1      AUC
  ────────────────────────────────────────
  XGBoost          0.7176   0.7206   0.7944
  RandomForest     0.7412   0.7478   0.7583

  Best: XGBoost  AUC=0.7944

  Computing SHAP for clinical expert...
  Top clinical SHAP features: ['Neuro TScore VarSE', 'ACC__fft_coefficient__attr_"real"__coeff_53', 'ACC__fft_coefficient__attr_"real"__coeff_39', 'General TScore Commissions', 'ACC__fft_coefficient__attr_"real"__coeff_56']


In [9]:
# =============================================================================
# CELL 4 — CLINICAL NEURAL ENCODER
# =============================================================================
print("\n  Training clinical neural encoder...")

class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, embed_dim))
        self.classifier = nn.Linear(embed_dim, 2)

    def get_embedding(self, x): return self.encoder(x)
    def forward(self, x):
        emb = self.encoder(x)
        return self.classifier(emb), emb

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clin_sel, y_clinical, test_size=0.2,
    stratify=y_clinical, random_state=42)
sc_c     = StandardScaler()
X_tr_c_s = sc_c.fit_transform(X_tr_c)
X_te_c_s = sc_c.transform(X_te_c)

X_tr_ct = torch.FloatTensor(X_tr_c_s).to(device)
y_tr_ct = torch.LongTensor(y_tr_c).to(device)
X_te_ct = torch.FloatTensor(X_te_c_s).to(device)
y_te_ct = torch.LongTensor(y_te_c).to(device)

cw_c       = get_class_weights(y_tr_c, device)
model_clinical = ClinicalEncoder(X_tr_c_s.shape[1]).to(device)
opt_c      = optim.Adam(model_clinical.parameters(), lr=1e-3, weight_decay=1e-4)
sched_c    = optim.lr_scheduler.ReduceLROnPlateau(opt_c,'max',factor=0.5,patience=8)
dl_c       = DataLoader(TensorDataset(X_tr_ct, y_tr_ct), batch_size=16,
                        sampler=make_weighted_sampler(y_tr_c))

train_nn(model_clinical, dl_c, X_te_ct, y_te_ct,
         criterion=nn.CrossEntropyLoss(weight=cw_c),
         optimizer=opt_c, scheduler=sched_c,
         max_epochs=150, patience=20, verbose_every=30, tag='Clinical')

X_all_c_s = sc_c.transform(X_clin_sel)
X_all_ct  = torch.FloatTensor(X_all_c_s).to(device)
model_clinical.eval()
with torch.no_grad():
    _, clinical_embeddings = model_clinical(X_all_ct)
clinical_embeddings = clinical_embeddings.detach().cpu()
clinical_labels     = y_clinical.copy()
print(f"✅ Clinical embeddings: {clinical_embeddings.shape}")



  Training clinical neural encoder...
  [Clinical] Early stop ep 25  best_f1=0.8235
✅ Clinical embeddings: torch.Size([85, 128])


In [ ]:
# =============================================================================
# CELL 5 — ACTIVITY/HRV BRANCH (richer features, same HYPERAKTIV subjects)
# =============================================================================
print("\n" + "="*65)
print("ACTIVITY/HRV BRANCH — HYPERAKTIV (richer features)")
print("="*65)

def extract_rich_ts_features(filepath, prefix):
    feats = {}
    try:
        df_ts = None
        for sep in [';', ',']:
            try:
                df_ts = pd.read_csv(filepath, sep=sep, skiprows=2)
                if df_ts.shape[1] > 1:
                    break
            except Exception:
                pass
        if df_ts is None:
            return {}
        num_cols = df_ts.select_dtypes(include=[np.number]).columns
        if len(num_cols) == 0:
            return {}
        sig = df_ts[num_cols[-1]].dropna().values.astype(float)
        if len(sig) < 20:
            return {}
        feats[f'{prefix}mean']        = float(np.mean(sig))
        feats[f'{prefix}std']         = float(np.std(sig))
        feats[f'{prefix}median']      = float(np.median(sig))
        feats[f'{prefix}range']       = float(np.ptp(sig))
        feats[f'{prefix}skew']        = float(skew(sig))
        feats[f'{prefix}kurtosis']    = float(kurtosis(sig))
        feats[f'{prefix}iqr']         = float(np.percentile(sig,75)
                                               - np.percentile(sig,25))
        feats[f'{prefix}p10']         = float(np.percentile(sig,10))
        feats[f'{prefix}p90']         = float(np.percentile(sig,90))
        feats[f'{prefix}cv']          = float(np.std(sig)/(np.mean(sig)+1e-8))
        above = (sig > np.mean(sig)).astype(int)
        feats[f'{prefix}transitions'] = float(np.sum(np.diff(above)!=0))
        if 'act' in prefix:
            thr    = np.percentile(sig, 25)
            active = (sig > thr).astype(int)
            starts = np.where(np.diff(np.concatenate([[0],active]))==1)[0]
            ends   = np.where(np.diff(np.concatenate([active,[0]]))==-1)[0]
            bouts  = ends - starts + 1
            if len(bouts) > 0:
                feats[f'{prefix}n_bouts']         = float(len(bouts))
                feats[f'{prefix}mean_bout_len']   = float(np.mean(bouts))
                feats[f'{prefix}short_bout_frac'] = float(
                    np.sum(bouts<=3)/(len(bouts)+1e-8))
        if 'hr' in prefix:
            diffs = np.diff(sig)
            feats[f'{prefix}rmssd']       = float(np.sqrt(np.mean(diffs**2)))
            feats[f'{prefix}sdnn']        = float(np.std(sig))
            feats[f'{prefix}pnn50_proxy'] = float(
                np.sum(np.abs(diffs) > np.percentile(np.abs(diffs),50))
                / (len(diffs)+1e-8))
            if len(sig) >= 64:
                try:
                    freqs, psd = sp_welch(sig, nperseg=min(64,len(sig)//2))
                    lf = (freqs>=0.04)&(freqs<0.15)
                    hf = (freqs>=0.15)&(freqs<0.40)
                    feats[f'{prefix}lf_hf_ratio'] = float(
                        psd[lf].sum()/(psd[hf].sum()+1e-8))
                except Exception:
                    pass
    except Exception:
        pass
    return feats


def safe_auc(y_true, probs):
    try:
        return float(roc_auc_score(y_true, probs))
    except Exception:
        return 0.5


def safe_auc_flipped(y_true, probs):
    try:
        return float(roc_auc_score(y_true, 1.0 - probs))
    except Exception:
        return 0.5


class BioEncoder(nn.Module):
    def __init__(self, input_dim, embed_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim,128), nn.BatchNorm1d(128),
            nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, embed_dim))
        self.classifier = nn.Linear(embed_dim, 2)
    def get_embedding(self, x): return self.encoder(x)
    def forward(self, x):
        emb = self.encoder(x)
        return self.classifier(emb), emb


y_bio_all = patient_info.set_index('ID')['ADHD'].to_dict()
all_bio   = {}
for dpath, prefix in [(os.path.join(HP,'activity_data'),'act_'),
                      (os.path.join(HP,'hrv_data'),     'hr_')]:
    if not os.path.exists(dpath):
        continue
    for fn in sorted(os.listdir(dpath)):
        if not fn.endswith('.csv'):
            continue
        m = re.search(r'(\d+)', fn)
        if not m:
            continue
        pid   = int(m.group(1))
        feats = extract_rich_ts_features(os.path.join(dpath,fn), prefix)
        if feats:
            if pid not in all_bio:
                all_bio[pid] = {}
            all_bio[pid].update(feats)

bio_rows = []
for pid, feats in all_bio.items():
    if pid in y_bio_all:
        row = dict(feats)
        row['ID']   = pid
        row['ADHD'] = y_bio_all[pid]
        bio_rows.append(row)

bio_df = pd.DataFrame(bio_rows).fillna(0)
if bio_df.empty:
    raise RuntimeError('No activity_hrv rows were extracted from HYPERAKTIV')

bio_cols        = [c for c in bio_df.columns if c not in ['ID','ADHD']]
X_bio_raw       = bio_df[bio_cols].values.astype(float)
X_bio_raw       = np.nan_to_num(X_bio_raw, nan=0, posinf=0, neginf=0)
y_bio           = bio_df['ADHD'].values.astype(int)
bio_subject_ids = bio_df['ID'].values.astype(int)

print(f"  Subjects: {len(y_bio)}  ADHD={sum(y_bio==1)}  HC={sum(y_bio==0)}")
print(f"  Features: {X_bio_raw.shape[1]}")

bio_benchmark_rows = []
xgb_bio_base = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.03,
    subsample=0.85, colsample_bytree=0.85,
    random_state=42, eval_metric='logloss', **xgb_hw)

# XGBoost 5-fold CV benchmark
bio_cv_preds_xgb = np.zeros(len(y_bio), dtype=int)
bio_cv_probs_xgb = np.zeros(len(y_bio), dtype=float)
for tri, vai in cv5.split(X_bio_raw, y_bio):
    sc_tmp = StandardScaler()
    Xtr_b  = sc_tmp.fit_transform(X_bio_raw[tri])
    Xva_b  = sc_tmp.transform(X_bio_raw[vai])
    m_bio  = deepcopy(xgb_bio_base)
    m_bio.fit(Xtr_b, y_bio[tri])
    bio_cv_preds_xgb[vai] = m_bio.predict(Xva_b)
    bio_cv_probs_xgb[vai] = m_bio.predict_proba(Xva_b)[:, 1]

bio_benchmark_rows.append({
    'model': 'XGBoost',
    'acc': float(accuracy_score(y_bio, bio_cv_preds_xgb)),
    'f1': float(f1_score(y_bio, bio_cv_preds_xgb, zero_division=0)),
    'auc': safe_auc(y_bio, bio_cv_probs_xgb),
    'auc_flipped': safe_auc_flipped(y_bio, bio_cv_probs_xgb),
})

# BioEncoder 5-fold CV benchmark
bio_cv_preds_nn = np.zeros(len(y_bio), dtype=int)
bio_cv_probs_nn = np.zeros(len(y_bio), dtype=float)
for fold, (tri, vai) in enumerate(cv5.split(X_bio_raw, y_bio), start=1):
    sc_tmp = StandardScaler()
    Xtr_b  = sc_tmp.fit_transform(X_bio_raw[tri])
    Xva_b  = sc_tmp.transform(X_bio_raw[vai])

    Xtr_bt = torch.FloatTensor(Xtr_b).to(device)
    ytr_bt = torch.LongTensor(y_bio[tri]).to(device)
    Xva_bt = torch.FloatTensor(Xva_b).to(device)
    yva_bt = torch.LongTensor(y_bio[vai]).to(device)

    cw_b_cv    = get_class_weights(y_bio[tri], device)
    model_b_cv = BioEncoder(Xtr_b.shape[1]).to(device)
    opt_b_cv   = optim.Adam(model_b_cv.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_b_cv = optim.lr_scheduler.ReduceLROnPlateau(
        opt_b_cv,'max',factor=0.5,patience=6)
    dl_b_cv    = DataLoader(TensorDataset(Xtr_bt,ytr_bt),
                            batch_size=min(16,len(tri)),
                            sampler=make_weighted_sampler(y_bio[tri]))

    train_nn(model_b_cv, dl_b_cv, Xva_bt, yva_bt,
             criterion=nn.CrossEntropyLoss(weight=cw_b_cv),
             optimizer=opt_b_cv, scheduler=sched_b_cv,
             max_epochs=80, patience=12, verbose_every=40,
             tag=f'BioCV{fold}')

    model_b_cv.eval()
    with torch.no_grad():
        lv_b, _ = model_b_cv(Xva_bt)
        bio_cv_preds_nn[vai] = lv_b.argmax(1).cpu().numpy()
        bio_cv_probs_nn[vai] = torch.softmax(lv_b,1)[:,1].cpu().numpy()

bio_benchmark_rows.append({
    'model': 'BioEncoder',
    'acc': float(accuracy_score(y_bio, bio_cv_preds_nn)),
    'f1': float(f1_score(y_bio, bio_cv_preds_nn, zero_division=0)),
    'auc': safe_auc(y_bio, bio_cv_probs_nn),
    'auc_flipped': safe_auc_flipped(y_bio, bio_cv_probs_nn),
})

bio_benchmark_metadata = {
    'dataset_scope': 'all_available_activity_hrv_subjects_before_alignment',
    'subject_count': int(len(y_bio)),
    'feature_count': int(len(bio_cols)),
    'feature_names': list(bio_cols),
    'cv_folds': int(cv5.get_n_splits()),
    'models': [
        {
            'model': row['model'],
            'acc': float(row['acc']),
            'f1': float(row['f1']),
            'auc': float(row['auc']),
            'auc_flipped': float(row['auc_flipped']),
        }
        for row in bio_benchmark_rows
    ],
}

print(f"\n  {'Model':<12} {'Acc':>8} {'F1':>8} {'AUC':>8} {'AUC_flip':>10}")
print("  " + "─"*54)
for row in bio_benchmark_rows:
    print(f"  {row['model']:<12} {row['acc']:>8.4f} {row['f1']:>8.4f} "
          f"{row['auc']:>8.4f} {row['auc_flipped']:>10.4f}")

best_bio_benchmark = max(bio_benchmark_rows, key=lambda row: row['auc_flipped'])
print(f"\n  Best flipped-aware benchmark: {best_bio_benchmark['model']} "
      f"(AUC={best_bio_benchmark['auc']:.4f}, "
      f"AUC_flipped={best_bio_benchmark['auc_flipped']:.4f})")

# Fit XGBoost on all extracted bio subjects for SHAP only
sc_bio_xgb_full = StandardScaler()
X_bio_xgb_full  = sc_bio_xgb_full.fit_transform(X_bio_raw)
xgb_bio         = deepcopy(xgb_bio_base)
xgb_bio.fit(X_bio_xgb_full, y_bio)

explainer_bio   = shap.TreeExplainer(xgb_bio)
shap_values_bio = explainer_bio.shap_values(X_bio_xgb_full)
sv_bio = (shap_values_bio[1] if isinstance(shap_values_bio,list)
          and len(shap_values_bio)>1 else
          shap_values_bio[0] if isinstance(shap_values_bio,list)
          else shap_values_bio)
shap_importance_bio = np.abs(sv_bio).mean(axis=0)
top_bio_idx  = np.argsort(shap_importance_bio)[::-1][:5]
print(f"  Top bio SHAP features: {[bio_cols[i] for i in top_bio_idx]}")
print("  Final exported BioEncoder will be retrained after clinical alignment.")


In [11]:
# =============================================================================
# CELL 6 — EEG BRANCH (DETEC-ADHD)
# =============================================================================
print("\n" + "="*65)
print("EEG BRANCH — DETEC-ADHD")
print("="*65)
print("Dataset: kaggle.com/datasets/danizo/eeg-dataset-for-adhd")
print()

EEG_PATH = PATHS['eeg']
eeg_csvs = glob.glob(os.path.join(EEG_PATH,'**','*.csv'), recursive=True)
adhdata  = [f for f in eeg_csvs if 'adhdata' in os.path.basename(f).lower()]
eeg_file = adhdata[0] if adhdata else eeg_csvs[0]

df_eeg   = pd.read_csv(eeg_file)
print(f"  Loaded: {os.path.basename(eeg_file)}  shape={df_eeg.shape}")

id_col    = next((c for c in df_eeg.columns if c.upper() in ('ID','SUBJECT')), None)
class_col = next((c for c in df_eeg.columns if c.upper() in ('CLASS','LABEL','ADHD')), None)
if id_col is None:
    df_eeg['_ID'] = (df_eeg.index // 256).astype(int); id_col = '_ID'
if class_col is None:
    raise ValueError("Cannot find class column in EEG CSV.")

eeg_channels = [c for c in df_eeg.columns if c not in [id_col,class_col]]
print(f"  Subjects: {df_eeg[id_col].nunique()}  Channels: {len(eeg_channels)}")

subjects_data, subjects_labels = [], []
for sid, grp in df_eeg.groupby(id_col):
    ed  = grp[eeg_channels].values.T.astype(np.float32)
    raw = str(grp[class_col].iloc[0]).upper()
    lab = 1 if ('ADHD' in raw and 'NON' not in raw
                 and 'CONTROL' not in raw) else 0
    subjects_data.append(ed); subjects_labels.append(lab)
subjects_labels = np.array(subjects_labels)
print(f"  ADHD={sum(subjects_labels==1)}  HC={sum(subjects_labels==0)}")

TGT = 1024
std_data = []
for rec in subjects_data:
    nc, nt = rec.shape
    if nt > TGT:
        s = (nt-TGT)//2; rec = rec[:,s:s+TGT]
    elif nt < TGT:
        rec = np.pad(rec,((0,0),(0,TGT-nt)))
    for ch in range(nc):
        m,s = rec[ch].mean(), rec[ch].std()
        if s > 0: rec[ch] = (rec[ch]-m)/s
    std_data.append(rec)

SEG = 512; OVL = 0.5
uniq_s = np.arange(len(std_data))
tr_s, te_s = train_test_split(uniq_s, test_size=0.2,
                               stratify=subjects_labels, random_state=42)
segments, seg_labels, seg_sids = [], [], []
for si, (rec,lab) in enumerate(zip(std_data,subjects_labels)):
    nc,tot = rec.shape; step = int(SEG*(1-OVL)); added = 0
    for st in range(0,tot-SEG+1,step):
        segments.append(rec[:,st:st+SEG])
        seg_labels.append(lab); seg_sids.append(si); added += 1
    if added == 0:
        pad = np.zeros((nc,SEG),dtype=np.float32)
        pad[:,:min(tot,SEG)] = rec[:,:min(tot,SEG)]
        segments.append(pad); seg_labels.append(lab); seg_sids.append(si)

segments   = np.array(segments,  dtype=np.float32)
seg_labels = np.array(seg_labels,dtype=np.int64)
seg_sids   = np.array(seg_sids,  dtype=np.int64)
tr_mask    = np.isin(seg_sids,tr_s); te_mask = np.isin(seg_sids,te_s)
X_eeg_tr, y_eeg_tr = segments[tr_mask], seg_labels[tr_mask]
X_eeg_te, y_eeg_te = segments[te_mask], seg_labels[te_mask]
mu_e = X_eeg_tr.mean(); sd_e = X_eeg_tr.std()
X_eeg_tr_n = (X_eeg_tr-mu_e)/(sd_e+1e-8)
X_eeg_te_n = (X_eeg_te-mu_e)/(sd_e+1e-8)
print(f"  Segments — train:{len(X_eeg_tr)}  test:{len(X_eeg_te)}")

class EEGDatasetAug(Dataset):
    def __init__(self, X, y, augment=False):
        self.X=X.copy(); self.y=torch.LongTensor(y); self.aug=augment
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        x = self.X[idx].copy()
        if self.aug:
            if random.random()>0.5: x+=np.random.randn(*x.shape).astype(np.float32)*0.08
            if random.random()>0.5: x=np.roll(x,random.randint(-20,20),axis=-1)
            if random.random()>0.5:
                for ch in range(x.shape[0]):
                    if random.random()<0.1: x[ch]=0.0
            if random.random()>0.5: x=x*random.uniform(0.8,1.2)
        return torch.FloatTensor(x).unsqueeze(0), self.y[idx]

N_CH = X_eeg_tr.shape[1]; N_T = X_eeg_tr.shape[2]

class EEGNet(nn.Module):
    def __init__(self,n_ch=N_CH,n_time=N_T,F1=8,D=2,F2=16,
                 dropout=0.5,embed_dim=128):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(1,F1,(1,64),padding=(0,32),bias=False),
            nn.BatchNorm2d(F1))
        self.conv2 = nn.Sequential(
            nn.Conv2d(F1,F1*D,(n_ch,1),groups=F1,bias=False),
            nn.BatchNorm2d(F1*D),nn.ELU(),nn.AvgPool2d((1,4)),nn.Dropout(dropout))
        self.conv3 = nn.Sequential(
            nn.Conv2d(F1*D,F2,(1,16),padding=(0,8),groups=F1*D,bias=False),
            nn.Conv2d(F2,F2,(1,1),bias=False),
            nn.BatchNorm2d(F2),nn.ELU(),nn.AvgPool2d((1,8)),nn.Dropout(dropout))
        with torch.no_grad():
            x_p = torch.zeros(1,1,n_ch,n_time)
            self._fs = self.conv3(self.conv2(self.conv1(x_p))).view(1,-1).size(1)
        self.embedding  = nn.Linear(self._fs, embed_dim)
        self.classifier = nn.Linear(embed_dim, 2)
    def get_embedding(self,x):
        x=self.conv1(x);x=self.conv2(x);x=self.conv3(x)
        return self.embedding(x.view(x.size(0),-1))
    def forward(self,x):
        emb=self.get_embedding(x)
        return self.classifier(emb),emb

train_ds = EEGDatasetAug(X_eeg_tr_n,y_eeg_tr,augment=True)
loader_eeg = DataLoader(train_ds,batch_size=64,
                        sampler=make_weighted_sampler(y_eeg_tr))
X_te_eeg_t = torch.FloatTensor(
    X_eeg_te_n.reshape(len(X_eeg_te_n),1,N_CH,N_T)).to(device)
y_te_eeg_t = torch.LongTensor(y_eeg_te).to(device)

model_eeg  = EEGNet(N_CH,N_T).to(device)
cw_e       = get_class_weights(y_eeg_tr,device)
opt_e      = optim.Adam(model_eeg.parameters(),lr=1e-3,weight_decay=1e-4)
sc_e       = optim.lr_scheduler.CosineAnnealingLR(opt_e,T_max=100)
es_eeg     = EarlyStopping(patience=15,mode='max')
crit_eeg   = nn.CrossEntropyLoss(weight=cw_e)

print("\n  Training EEGNet...")
for ep in range(100):
    model_eeg.train(); tl=0; nb=0
    for xb,yb in loader_eeg:
        xb,yb = xb.to(device),yb.to(device)
        opt_e.zero_grad()
        logits,_ = model_eeg(xb)
        loss = crit_eeg(logits,yb)
        loss.backward(); opt_e.step()
        tl+=loss.item(); nb+=1
    sc_e.step()
    model_eeg.eval()
    with torch.no_grad():
        lv,_ = model_eeg(X_te_eeg_t)
        pv   = lv.argmax(1).cpu().numpy()
        f1v  = f1_score(y_eeg_te,pv,zero_division=0)
        try: auv = roc_auc_score(y_eeg_te,
                                  torch.softmax(lv,1)[:,1].cpu().numpy())
        except: auv=0.5
    if (ep+1)%20==0:
        print(f"  [EEGNet] Ep {ep+1:3d}  loss={tl/nb:.4f}  "
              f"f1={f1v:.4f}  auc={auv:.4f}")
    if es_eeg.step(f1v,model_eeg): break
es_eeg.restore(model_eeg)

# Subject-level embeddings (mean over segments)
all_segs_t = torch.FloatTensor(
    segments.reshape(len(segments),1,N_CH,N_T))
all_norm   = (all_segs_t - mu_e)/(sd_e+1e-8)
model_eeg.eval()
emb_segs = []
with torch.no_grad():
    for s in range(0,len(all_norm),256):
        _,emb = model_eeg(all_norm[s:s+256].to(device))
        emb_segs.append(emb.cpu())
emb_segs = torch.cat(emb_segs)

eeg_emb_list, eeg_lab_list = [], []
for si in range(len(std_data)):
    mask = seg_sids == si
    if mask.sum()>0:
        eeg_emb_list.append(emb_segs[mask].mean(0))
        eeg_lab_list.append(subjects_labels[si])
eeg_embeddings = torch.stack(eeg_emb_list)
eeg_labels     = np.array(eeg_lab_list)
print(f"✅ EEG embeddings: {eeg_embeddings.shape}  "
      f"ADHD={sum(eeg_labels==1)}  HC={sum(eeg_labels==0)}")

# SHAP for EEG
print("\n  Computing SHAP for EEGNet...")
bg_idx  = np.random.choice(len(X_eeg_tr_n),min(50,len(X_eeg_tr_n)),replace=False)
bg_data = torch.FloatTensor(
    X_eeg_tr_n[bg_idx].reshape(len(bg_idx),1,N_CH,N_T)).to(device)

class EEGNetLogits(nn.Module):
    def __init__(self,m): super().__init__(); self.m=m
    def forward(self,x):
        logits,_ = self.m(x); return logits

explainer_eeg = shap.DeepExplainer(EEGNetLogits(model_eeg), bg_data)
te_sample     = torch.FloatTensor(
    X_eeg_te_n[:min(30,len(X_eeg_te_n))].reshape(-1,1,N_CH,N_T)).to(device)
shap_values_eeg = explainer_eeg.shap_values(te_sample)

sv = np.array(shap_values_eeg)
print(f"  SHAP raw shape: {sv.shape}")
# Robustly extract ADHD-class channel importance
if sv.ndim == 5:
    sv_adhd = sv[..., 1] if sv.shape[-1] == 2 else sv[..., 0]
    sv_adhd = sv_adhd.squeeze(1)  # (samples, ch, time)
elif sv.ndim == 4:
    sv_adhd = sv.squeeze(1)
else:
    sv_adhd = sv.reshape(sv.shape[0], N_CH, -1)
channel_importance_eeg = np.abs(sv_adhd).mean(axis=(0,-1))
top_eeg_channels = np.argsort(channel_importance_eeg)[::-1][:5]
print(f"  Top EEG channels: "
      f"{[eeg_channels[i] for i in top_eeg_channels]}")



EEG BRANCH — DETEC-ADHD
Dataset: kaggle.com/datasets/danizo/eeg-dataset-for-adhd

  Loaded: adhdata.csv  shape=(2166383, 21)
  Subjects: 121  Channels: 19
  ADHD=61  HC=60
  Segments — train:288  test:75

  Training EEGNet...
  [EEGNet] Ep  20  loss=0.4986  f1=0.6835  auc=0.7172
  [EEGNet] Ep  40  loss=0.4290  f1=0.7500  auc=0.7991
  [EEGNet] Ep  60  loss=0.2473  f1=0.8101  auc=0.8469
✅ EEG embeddings: torch.Size([121, 128])  ADHD=61  HC=60

  Computing SHAP for EEGNet...
  SHAP raw shape: (30, 1, 19, 512, 2)
  Top EEG channels: ['O2', 'P7', 'P3', 'F7', 'Fp1']


In [12]:
import glob, os, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, roc_auc_score
from copy import deepcopy
import xgboost as xgb
import shap

# ── Device ────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  Device: {device}")

# ── Utilities ─────────────────────────────────────────────────────────────
def get_class_weights(y, dev):
    cw = compute_class_weight('balanced', classes=np.unique(y), y=y)
    return torch.FloatTensor(cw).to(dev)

def make_weighted_sampler(y):
    counts  = np.bincount(y)
    weights = 1.0 / counts[y]
    return WeightedRandomSampler(torch.FloatTensor(weights), len(weights))

class EarlyStopping:
    def __init__(self, patience=12, mode='max'):
        self.patience = patience; self.mode = mode
        self.best = None; self.bad = 0; self.best_state = None
    def step(self, val, model):
        improved = (self.best is None
                    or (self.mode == 'max' and val > self.best)
                    or (self.mode == 'min' and val < self.best))
        if improved:
            self.best = val; self.bad = 0
            self.best_state = deepcopy(model.state_dict())
        else:
            self.bad += 1
        return self.bad >= self.patience
    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)

def train_nn(model, loader, X_val, y_val, criterion, optimizer,
             scheduler, max_epochs=150, patience=15,
             verbose_every=20, tag='Model'):
    es = EarlyStopping(patience=patience, mode='max')
    for ep in range(max_epochs):
        model.train(); total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out    = model(xb)
            logits = out[0] if isinstance(out, tuple) else out
            loss   = criterion(logits, yb)
            loss.backward(); optimizer.step()
            total_loss += loss.item()
        model.eval()
        with torch.no_grad():
            out_v  = model(X_val)
            lv     = out_v[0] if isinstance(out_v, tuple) else out_v
            pv     = lv.argmax(1).cpu().numpy()
            f1v    = f1_score(y_val.cpu().numpy(), pv, zero_division=0)
            try:    scheduler.step(f1v)
            except: scheduler.step()
        if (ep + 1) % verbose_every == 0:
            print(f"  [{tag}] Ep {ep+1:3d}  "
                  f"loss={total_loss/len(loader):.4f}  val_f1={f1v:.4f}")
        if es.step(f1v, model):
            print(f"  [{tag}] Early stop ep {ep+1}  best_f1={es.best:.4f}")
            break
    es.restore(model)
    return model

xgb_v  = tuple(int(x) for x in xgb.__version__.split('.')[:2])
xgb_hw = ({'tree_method': 'hist', 'device': 'cuda'}
           if (torch.cuda.is_available() and xgb_v >= (2, 0))
           else {'tree_method': 'hist'})

# =============================================================================
print("\n" + "="*65)
print("BRAIN BRANCH — ADHD-200 Preprocessed")
print("="*65)
print("Dataset: kaggle.com/datasets/kishore00afk/adhd-200-preprocessed")
print()

ADHD200_PATH = '/kaggle/input/datasets/kishore00afk/adhd-200-preprocessed'
PHENO_FILE   = os.path.join(ADHD200_PATH,
                   'NITRC-multi-file-downloads',
                   'adhd200_preprocessed_phenotypics.tsv')
TC_ROOT      = os.path.join(ADHD200_PATH,
                   'ADHD200_CC200_TCs',
                   'ADHD200_CC200_TCs_filtfix')

# Initialise all outputs to None
brain_available       = False
brain_embeddings      = None
brain_labels          = None
model_brain           = None
X_te_b2               = None
y_te_b2               = None
X_te_b2t              = None
y_te_b2t              = None
shap_importance_brain = None

# ── Step 1: Load phenotypic file ──────────────────────────────────────────
print(f"  Phenotypic: {os.path.basename(PHENO_FILE)}")
pheno_df = pd.read_csv(PHENO_FILE, sep='\t')
print(f"  Shape: {pheno_df.shape}")

id_col_b = None
for c in ['ScanDir ID', 'Subject', 'subject', 'participant_id', 'ID']:
    if c in pheno_df.columns:
        id_col_b = c; break
if id_col_b is None:
    id_col_b = pheno_df.columns[0]

lab_col_b = None
for c in ['DX', 'dx', 'ADHD', 'Diagnosis', 'diagnosis', 'dx_group']:
    if c in pheno_df.columns:
        lab_col_b = c; break
if lab_col_b is None:
    num_cols  = pheno_df.select_dtypes(include=[np.number]).columns.tolist()
    lab_col_b = num_cols[1] if len(num_cols) > 1 else num_cols[0]

print(f"  ID='{id_col_b}'  Label='{lab_col_b}'")
y_pheno = (pd.to_numeric(pheno_df[lab_col_b],
                          errors='coerce').fillna(0) > 0).astype(int)
pheno_df['_y']   = y_pheno.values
pheno_df['_sid'] = pheno_df[id_col_b].astype(str).str.strip().str.lstrip('0')
print(f"  ADHD={sum(y_pheno==1)}  HC={sum(y_pheno==0)}")

# ── Step 2: Find .1D files ────────────────────────────────────────────────
conn_files = (
    glob.glob(os.path.join(TC_ROOT, '**', '*.1D'), recursive=True) +
    glob.glob(os.path.join(TC_ROOT, '**', '*.1d'), recursive=True))
conn_files = [f for f in conn_files
              if os.path.basename(f).startswith('sfnwmrda')]
print(f"  Found {len(conn_files)} sfnwmrda .1D files")

# ── Step 3: Compute FC features per subject ───────────────────────────────
records = []; failed = 0
for fpath in conn_files:
    sid = os.path.basename(os.path.dirname(fpath)).lstrip('0')
    try:
        tc = pd.read_csv(fpath, sep=r'\s+', header=None,
                         comment='#', engine='python')
        tc = tc.apply(pd.to_numeric, errors='coerce').dropna(how='all')
        if tc.shape[0] < tc.shape[1]: tc = tc.T
        if tc.shape[0] < 10 or tc.shape[1] < 2:
            failed += 1; continue
        corr = tc.corr().values
        np.fill_diagonal(corr, 0)
        corr   = np.nan_to_num(corr, nan=0, posinf=0, neginf=0)
        corr_z = np.arctanh(np.clip(corr, -0.999, 0.999))
        idx    = np.triu_indices(corr_z.shape[0], k=1)
        feat   = corr_z[idx].astype(np.float32)
        records.append({'_sid': sid,
                        **{f'fc_{i}': float(v) for i, v in enumerate(feat)}})
    except Exception:
        failed += 1

print(f"  Processed: {len(records)}  Failed: {failed}")

if len(records) == 0:
    print("  No subjects processed — brain branch skipped.")
else:
    conn_df = pd.DataFrame(records)
    conn_df['_sid'] = conn_df['_sid'].astype(str).str.strip()

    # ── Step 4: Merge labels with FC features ─────────────────────────────
    pheno_df['_sid_n'] = pheno_df['_sid'].str.lstrip('0')
    conn_df['_sid_n']  = conn_df['_sid'].str.lstrip('0')

    merged_b = pd.merge(
        pheno_df[['_sid_n', lab_col_b, '_y']],
        conn_df, on='_sid_n', how='inner')
    merged_b = (merged_b
                .drop_duplicates(subset=['_sid_n'], keep='first')
                .reset_index(drop=True))
    print(f"  Merged+dedup: {len(merged_b)} subjects")

    if len(merged_b) == 0:
        print("  Merge produced 0 rows — brain branch skipped.")
        print(f"  Pheno IDs sample: {pheno_df['_sid_n'].head(5).tolist()}")
        print(f"  .1D  IDs sample:  {conn_df['_sid_n'].head(5).tolist()}")
    else:
        y_brain = merged_b['_y'].values.astype(int)
        print(f"  ADHD={sum(y_brain==1)}  HC={sum(y_brain==0)}")

        fc_cols = [c for c in merged_b.columns if c.startswith('fc_')]
        X_brain = merged_b[fc_cols].values.astype(np.float32)
        X_brain = np.nan_to_num(X_brain, nan=0, posinf=0, neginf=0)
        print(f"  Feature matrix: {X_brain.shape}  — filtering + PCA...")

        # ── Step 5: Variance filter BEFORE PCA (key speed fix) ───────────
        # Drops ~70% of near-zero-variance columns (failed/constant
        # correlations) before PCA touches the data.
        # 768 x 18336  →  768 x ~5500  →  PCA runs in ~30s not 10min.
        col_var       = np.var(X_brain, axis=0)
        var_threshold = np.percentile(col_var, 70)  # keep top 30%
        X_brain_filt  = X_brain[:, col_var > var_threshold]
        print(f"  Variance filter: {X_brain.shape[1]} → "
              f"{X_brain_filt.shape[1]} features (kept top 30% by variance)")

        # ── Step 6: Randomized PCA ────────────────────────────────────────
        n_comp      = min(100, len(y_brain) // 5, X_brain_filt.shape[1])
        pca_b       = PCA(n_components=n_comp, svd_solver='randomized',
                          random_state=42)
        X_brain_pca = pca_b.fit_transform(X_brain_filt)
        print(f"  PCA: {X_brain_filt.shape[1]} → {n_comp}  "
              f"(var explained: {pca_b.explained_variance_ratio_.sum():.3f})")

        # ── Step 7: Train / test split ────────────────────────────────────
        X_tr_b2, X_te_b2, y_tr_b2, y_te_b2 = train_test_split(
            X_brain_pca, y_brain,
            test_size=0.2, stratify=y_brain, random_state=42)

        sc_b2     = StandardScaler()
        X_tr_b2_s = sc_b2.fit_transform(X_tr_b2)
        X_te_b2_s = sc_b2.transform(X_te_b2)

        X_tr_b2t = torch.FloatTensor(X_tr_b2_s).to(device)
        y_tr_b2t = torch.LongTensor(y_tr_b2).to(device)
        X_te_b2t = torch.FloatTensor(X_te_b2_s).to(device)
        y_te_b2t = torch.LongTensor(y_te_b2).to(device)

        # ── Step 8: Brain Encoder ─────────────────────────────────────────
        class BrainEncoder(nn.Module):
            def __init__(self, input_dim, embed_dim=128):
                super().__init__()
                self.encoder = nn.Sequential(
                    nn.Linear(input_dim, 256),
                    nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
                    nn.Linear(256, 128),
                    nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
                    nn.Linear(128, embed_dim))
                self.classifier = nn.Linear(embed_dim, 2)
            def get_embedding(self, x): return self.encoder(x)
            def forward(self, x):
                emb = self.encoder(x)
                return self.classifier(emb), emb

        cw_b2       = get_class_weights(y_tr_b2, device)
        model_brain = BrainEncoder(n_comp).to(device)
        opt_b2      = optim.Adam(model_brain.parameters(),
                                 lr=1e-3, weight_decay=1e-4)
        sc_b2s      = optim.lr_scheduler.ReduceLROnPlateau(
                          opt_b2, 'max', factor=0.5, patience=8)
        dl_b2       = DataLoader(
                          TensorDataset(X_tr_b2t, y_tr_b2t),
                          batch_size=min(64, len(y_tr_b2)),
                          sampler=make_weighted_sampler(y_tr_b2))

        print("\n  Training Brain Encoder...")
        train_nn(model_brain, dl_b2, X_te_b2t, y_te_b2t,
                 criterion=nn.CrossEntropyLoss(weight=cw_b2),
                 optimizer=opt_b2, scheduler=sc_b2s,
                 max_epochs=80, patience=12,
                 verbose_every=20, tag='Brain')

        # ── Step 9: Extract embeddings ────────────────────────────────────
        X_all_b2_s = sc_b2.transform(X_brain_pca)
        X_all_b2t  = torch.FloatTensor(X_all_b2_s).to(device)
        model_brain.eval()
        with torch.no_grad():
            _, brain_embeddings = model_brain(X_all_b2t)
        brain_embeddings = brain_embeddings.detach().cpu()
        brain_labels     = y_brain.copy()
        brain_available  = True

        print(f"\n✅ Brain embeddings: {brain_embeddings.shape}  "
              f"ADHD={sum(brain_labels==1)}  HC={sum(brain_labels==0)}")

        # ── Step 10: SHAP on XGBoost (PCA features) ──────────────────────
        print("\n  Computing SHAP for brain expert...")
        xgb_brain = xgb.XGBClassifier(
            n_estimators=200, max_depth=4,
            random_state=42, eval_metric='logloss', **xgb_hw)
        xgb_brain.fit(X_all_b2_s, y_brain)

        explainer_brain   = shap.TreeExplainer(xgb_brain)
        shap_values_brain = explainer_brain.shap_values(X_all_b2_s)

        sv_b = (shap_values_brain[1]
                if isinstance(shap_values_brain, list)
                   and len(shap_values_brain) > 1
                else shap_values_brain[0]
                if isinstance(shap_values_brain, list)
                else shap_values_brain)

        shap_importance_brain = np.abs(sv_b).mean(axis=0)
        top_b = np.argsort(shap_importance_brain)[::-1][:10]
        print(f"  Top PCA components: {top_b.tolist()}")
        print(f"  Importance values:  "
              f"{shap_importance_brain[top_b].round(4)}")

if not brain_available:
    print("\n  Brain branch skipped — fusion runs on remaining modalities.")

  Device: cuda

BRAIN BRANCH — ADHD-200 Preprocessed
Dataset: kaggle.com/datasets/kishore00afk/adhd-200-preprocessed

  Phenotypic: adhd200_preprocessed_phenotypics.tsv
  Shape: (973, 19)
  ID='ScanDir ID'  Label='DX'
  ADHD=362  HC=611
  Found 1198 sfnwmrda .1D files
  Processed: 1198  Failed: 0
  Merged+dedup: 768 subjects
  ADHD=280  HC=488
  Feature matrix: (768, 18336)  — filtering + PCA...
  Variance filter: 18336 → 5501 features (kept top 30% by variance)
  PCA: 5501 → 100  (var explained: 0.656)

  Training Brain Encoder...
  [Brain] Early stop ep 14  best_f1=0.5783

✅ Brain embeddings: torch.Size([768, 128])  ADHD=280  HC=488

  Computing SHAP for brain expert...
  Top PCA components: [3, 25, 12, 27, 46, 90, 50, 52, 36, 48]
  Importance values:  [0.6452 0.4894 0.436  0.3946 0.385  0.3071 0.2679 0.2558 0.2525 0.2404]


In [ ]:
# =============================================================================
# ALIGNMENT CELL — rebuild aligned bio dataset and final export expert
# =============================================================================
import numpy as np
from sklearn.preprocessing import StandardScaler

print("Rebuilding aligned activity_hrv dataset in clinical subject order...")

clinical_subject_order = merged['ID'].values.astype(int)
bio_df_indexed         = bio_df.set_index('ID')
missing_bio_ids        = [int(pid) for pid in clinical_subject_order
                          if pid not in bio_df_indexed.index]
if missing_bio_ids:
    raise ValueError(
        f"Missing {len(missing_bio_ids)} clinical subjects in bio_df: "
        f"{missing_bio_ids[:10]}")

bio_df_aligned          = bio_df_indexed.loc[clinical_subject_order].reset_index()
bio_cols_aligned        = [c for c in bio_cols if c in bio_df_aligned.columns]
X_bio_raw_aligned       = bio_df_aligned[bio_cols_aligned].values.astype(float)
X_bio_raw_aligned       = np.nan_to_num(X_bio_raw_aligned, nan=0, posinf=0, neginf=0)
y_bio_aligned           = bio_df_aligned['ADHD'].values.astype(int)
bio_subject_ids_aligned = bio_df_aligned['ID'].values.astype(int)
labels_match            = np.array_equal(y_bio_aligned, y_clinical)

print(f"  Clinical subjects           : {len(clinical_subject_order)}")
print(f"  Aligned bio subjects        : {len(bio_df_aligned)}")
print(f"  Aligned feature count       : {X_bio_raw_aligned.shape[1]}")
print(f"  Labels match clinical order : {labels_match}")

# Promote aligned activity_hrv data to the canonical downstream variables
bio_cols        = list(bio_cols_aligned)
X_bio_raw       = X_bio_raw_aligned
y_bio           = y_bio_aligned
bio_subject_ids = bio_subject_ids_aligned
bio_labels      = y_bio.copy()

paired_cv_splits = list(cv5.split(np.zeros(len(y_bio)), y_bio))

print("\nRunning aligned BioEncoder 5-fold benchmark on shared subjects...")
bio_aligned_oof_probs = np.zeros(len(y_bio), dtype=float)
bio_aligned_oof_preds = np.zeros(len(y_bio), dtype=int)
bio_aligned_oof_embeddings_np = None

for fold, (tri, vai) in enumerate(paired_cv_splits, start=1):
    set_seed(420 + fold)
    sc_tmp = StandardScaler()
    Xtr_b  = sc_tmp.fit_transform(X_bio_raw[tri])
    Xva_b  = sc_tmp.transform(X_bio_raw[vai])

    Xtr_bt = torch.FloatTensor(Xtr_b).to(device)
    ytr_bt = torch.LongTensor(y_bio[tri]).to(device)
    Xva_bt = torch.FloatTensor(Xva_b).to(device)
    yva_bt = torch.LongTensor(y_bio[vai]).to(device)

    cw_b_cv    = get_class_weights(y_bio[tri], device)
    model_b_cv = BioEncoder(Xtr_b.shape[1]).to(device)
    opt_b_cv   = optim.Adam(model_b_cv.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_b_cv = optim.lr_scheduler.ReduceLROnPlateau(
        opt_b_cv,'max',factor=0.5,patience=6)
    dl_b_cv    = DataLoader(TensorDataset(Xtr_bt,ytr_bt),
                            batch_size=min(16,len(tri)),
                            sampler=make_weighted_sampler(y_bio[tri]))

    train_nn(model_b_cv, dl_b_cv, Xva_bt, yva_bt,
             criterion=nn.CrossEntropyLoss(weight=cw_b_cv),
             optimizer=opt_b_cv, scheduler=sched_b_cv,
             max_epochs=80, patience=12, verbose_every=40,
             tag=f'BioAlignedCV{fold}')

    model_b_cv.eval()
    with torch.no_grad():
        lv_b, emb_b = model_b_cv(Xva_bt)
        if bio_aligned_oof_embeddings_np is None:
            bio_aligned_oof_embeddings_np = np.zeros(
                (len(y_bio), emb_b.shape[1]), dtype=np.float32)
        bio_aligned_oof_embeddings_np[vai] = emb_b.detach().cpu().numpy()
        bio_aligned_oof_preds[vai] = lv_b.argmax(1).cpu().numpy()
        bio_aligned_oof_probs[vai] = torch.softmax(lv_b,1)[:,1].cpu().numpy()

bio_aligned_oof_metrics = {
    'model': 'BioEncoder_aligned_oof',
    'subject_count': int(len(y_bio)),
    'cv_folds': int(len(paired_cv_splits)),
    'acc': float(accuracy_score(y_bio, bio_aligned_oof_preds)),
    'f1': float(f1_score(y_bio, bio_aligned_oof_preds, zero_division=0)),
    'auc': safe_auc(y_bio, bio_aligned_oof_probs),
    'auc_flipped': safe_auc_flipped(y_bio, bio_aligned_oof_probs),
}
bio_aligned_oof_embeddings = torch.FloatTensor(bio_aligned_oof_embeddings_np)
print(f"  Aligned BioEncoder OOF  Acc={bio_aligned_oof_metrics['acc']:.4f}  "
      f"F1={bio_aligned_oof_metrics['f1']:.4f}  "
      f"AUC={bio_aligned_oof_metrics['auc']:.4f}  "
      f"AUC_flip={bio_aligned_oof_metrics['auc_flipped']:.4f}")
print("  This aligned BioEncoder OOF AUC is the activity_hrv gate reference.")

print("\nTraining final exported BioEncoder on aligned subjects...")
set_seed(42)
sc_b        = StandardScaler()
X_bio_all_s = sc_b.fit_transform(X_bio_raw)
X_bio_all_t = torch.FloatTensor(X_bio_all_s).to(device)
y_bio_all_t = torch.LongTensor(y_bio).to(device)

cw_b_final    = get_class_weights(y_bio, device)
model_bio     = BioEncoder(X_bio_all_s.shape[1]).to(device)
opt_b_final   = optim.Adam(model_bio.parameters(), lr=1e-3, weight_decay=1e-4)
sched_b_final = optim.lr_scheduler.ReduceLROnPlateau(
    opt_b_final,'max',factor=0.5,patience=8)
dl_b_final    = DataLoader(TensorDataset(X_bio_all_t,y_bio_all_t),
                           batch_size=min(16,len(y_bio)),
                           sampler=make_weighted_sampler(y_bio))

train_nn(model_bio, dl_b_final, X_bio_all_t, y_bio_all_t,
         criterion=nn.CrossEntropyLoss(weight=cw_b_final),
         optimizer=opt_b_final, scheduler=sched_b_final,
         max_epochs=120, patience=18, verbose_every=30,
         tag='BioAlignedFinal')

activity_hrv_export_scaler = sc_b
activity_hrv_export_feature_names = list(bio_cols)
activity_hrv_export_subject_ids = [int(pid) for pid in bio_subject_ids]
activity_hrv_export_metadata = {
    'alignment': {
        'reference': 'clinical_subject_order_from_merged',
        'subject_count': int(len(activity_hrv_export_subject_ids)),
        'subject_ids': activity_hrv_export_subject_ids,
        'labels_match_clinical': bool(labels_match),
    },
    'final_model': {
        'model_type': 'BioEncoder',
        'embed_dim': 128,
        'input_dim': int(X_bio_raw.shape[1]),
        'trained_on_aligned_subjects_only': True,
    },
    'benchmark': bio_benchmark_metadata,
}

bio_embeddings = None
print("  ✅ Final activity_hrv expert is aligned and ready for fusion/export")


In [ ]:
# =============================================================================
# CELL 8 — MULTIMODAL FUSION + EVALUATION
# =============================================================================
print("\n" + "="*65)
print("MULTIMODAL FUSION")
print("="*65)

if 'paired_cv_splits' not in globals() or 'bio_aligned_oof_embeddings' not in globals():
    raise RuntimeError('Aligned BioEncoder OOF artifacts are required before fusion calibration.')

print("  Keeping the full-fit aligned BioEncoder for export only.")
print("  Building shared-fold clinical OOF embeddings for fusion calibration...")
clinical_oof_probs = np.zeros(len(y_clinical), dtype=float)
clinical_oof_preds = np.zeros(len(y_clinical), dtype=int)
clinical_oof_embeddings_np = None

for fold, (tri, vai) in enumerate(paired_cv_splits, start=1):
    set_seed(520 + fold)
    sc_tmp = StandardScaler()
    Xtr_c  = sc_tmp.fit_transform(X_clin_sel[tri])
    Xva_c  = sc_tmp.transform(X_clin_sel[vai])

    Xtr_ct = torch.FloatTensor(Xtr_c).to(device)
    ytr_ct = torch.LongTensor(y_clinical[tri]).to(device)
    Xva_ct = torch.FloatTensor(Xva_c).to(device)
    yva_ct = torch.LongTensor(y_clinical[vai]).to(device)

    cw_c_cv    = get_class_weights(y_clinical[tri], device)
    model_c_cv = ClinicalEncoder(Xtr_c.shape[1]).to(device)
    opt_c_cv   = optim.Adam(model_c_cv.parameters(), lr=1e-3, weight_decay=1e-4)
    sched_c_cv = optim.lr_scheduler.ReduceLROnPlateau(
        opt_c_cv,'max',factor=0.5,patience=6)
    dl_c_cv    = DataLoader(TensorDataset(Xtr_ct,ytr_ct),
                            batch_size=min(16,len(tri)),
                            sampler=make_weighted_sampler(y_clinical[tri]))

    train_nn(model_c_cv, dl_c_cv, Xva_ct, yva_ct,
             criterion=nn.CrossEntropyLoss(weight=cw_c_cv),
             optimizer=opt_c_cv, scheduler=sched_c_cv,
             max_epochs=80, patience=12, verbose_every=40,
             tag=f'ClinicalCV{fold}')

    model_c_cv.eval()
    with torch.no_grad():
        lv_c, emb_c = model_c_cv(Xva_ct)
        if clinical_oof_embeddings_np is None:
            clinical_oof_embeddings_np = np.zeros(
                (len(y_clinical), emb_c.shape[1]), dtype=np.float32)
        clinical_oof_embeddings_np[vai] = emb_c.detach().cpu().numpy()
        clinical_oof_preds[vai] = lv_c.argmax(1).cpu().numpy()
        clinical_oof_probs[vai] = torch.softmax(lv_c,1)[:,1].cpu().numpy()

clinical_oof_metrics = {
    'model': 'ClinicalEncoder_paired_oof',
    'subject_count': int(len(y_clinical)),
    'cv_folds': int(len(paired_cv_splits)),
    'acc': float(accuracy_score(y_clinical, clinical_oof_preds)),
    'f1': float(f1_score(y_clinical, clinical_oof_preds, zero_division=0)),
    'auc': safe_auc(y_clinical, clinical_oof_probs),
    'auc_flipped': safe_auc_flipped(y_clinical, clinical_oof_probs),
}

fusion_clinical_embeddings = torch.FloatTensor(clinical_oof_embeddings_np)
fusion_bio_embeddings = (bio_aligned_oof_embeddings.detach().cpu()
                         if isinstance(bio_aligned_oof_embeddings, torch.Tensor)
                         else torch.FloatTensor(np.array(bio_aligned_oof_embeddings)))
print(f"  Clinical encoder OOF   Acc={clinical_oof_metrics['acc']:.4f}  "
      f"F1={clinical_oof_metrics['f1']:.4f}  "
      f"AUC={clinical_oof_metrics['auc']:.4f}")
print(f"  Activity_hrv OOF ref   Acc={bio_aligned_oof_metrics['acc']:.4f}  "
      f"F1={bio_aligned_oof_metrics['f1']:.4f}  "
      f"AUC={bio_aligned_oof_metrics['auc']:.4f}")
print("  Using shared-fold OOF embeddings for paired-modality fusion evaluation.")

MODALITY_REGISTRY = [
    ('fusion_clinical_embeddings', 'clinical_labels', 'clinical'),
    ('fusion_bio_embeddings',      'bio_labels',       'activity_hrv'),
    ('eeg_embeddings',             'eeg_labels',       'eeg'),
    ('brain_embeddings',           'brain_labels',     'brain'),
]

available        = {}
available_labels = {}

for emb_var, lab_var, name in MODALITY_REGISTRY:
    emb = globals().get(emb_var,None)
    lab = globals().get(lab_var,None)
    if emb is None or lab is None:
        print(f"  ⚠️  {name:<14} skipped")
        continue
    if isinstance(emb,torch.Tensor): emb = emb.detach().cpu()
    else: emb = torch.FloatTensor(np.array(emb))
    lab = np.array(lab).astype(int)
    available[name]        = emb
    available_labels[name] = lab
    print(f"  ✅ {name:<14}  shape={tuple(emb.shape)}  "
          f"ADHD={sum(lab==1)}  HC={sum(lab==0)}")

mod_names = list(available.keys())
n_mods    = len(mod_names)
print(f"\n  Total modalities: {n_mods}")

paired_names = []
ref_n = None; ref_labels = None
for nm in ['clinical','activity_hrv']:
    if nm not in available: continue
    n = available[nm].shape[0]
    if ref_n is None:
        ref_n=n; ref_labels=available_labels[nm]; paired_names.append(nm)
    elif n == ref_n:
        paired_names.append(nm)

print(f"  Paired: {paired_names}")
print(f"  Benchmarks: {[n for n in mod_names if n not in paired_names]}")

assert paired_names == ['clinical','activity_hrv'], (
    'Need aligned clinical and activity_hrv modalities for calibrated fusion.')

class MoEFusion(nn.Module):
    def __init__(self,mod_dims,embed_dim=128,n_classes=2,temp=1.0):
        super().__init__()
        self.mod_names = list(mod_dims.keys())
        self.experts   = nn.ModuleDict({
            nm: nn.Sequential(nn.Linear(d,embed_dim),
                              nn.LayerNorm(embed_dim),
                              nn.ReLU(),nn.Dropout(0.2))
            for nm,d in mod_dims.items()})
        gate_in   = sum(mod_dims.values())
        self.gate = nn.Sequential(
            nn.Linear(gate_in,128),nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(128,len(mod_dims)))
        self.temp  = temp
        self.cross = nn.MultiheadAttention(
            embed_dim,num_heads=4,dropout=0.1,batch_first=True)
        self.cls   = nn.Sequential(
            nn.Linear(embed_dim,64),nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(64,n_classes))

    def forward(self,x_concat,return_gates=False):
        slices = {}; offset = 0
        for nm in self.mod_names:
            d = self.experts[nm][0].in_features
            slices[nm] = x_concat[:,offset:offset+d]; offset+=d
        expert_outs = torch.stack(
            [self.experts[nm](slices[nm]) for nm in self.mod_names],dim=1)
        gate_logits  = self.gate(x_concat)/self.temp
        gate_weights = torch.softmax(gate_logits,dim=-1).unsqueeze(-1)
        fused        = (expert_outs*gate_weights).sum(dim=1)
        attn_out,_   = self.cross(expert_outs,expert_outs,expert_outs)
        fused        = fused + attn_out.mean(dim=1)
        logits       = self.cls(fused)
        if return_gates: return logits, gate_weights.squeeze(-1)
        return logits


def aggregate_history(histories, n_modalities):
    if not histories:
        return {'train_loss':[], 'val_f1':[], 'val_auc':[], 'gates':[]}
    out = {}
    for key in ['train_loss','val_f1','val_auc']:
        max_len = max(len(hist[key]) for hist in histories)
        arr = np.full((len(histories), max_len), np.nan, dtype=float)
        for row, hist in enumerate(histories):
            arr[row, :len(hist[key])] = hist[key]
        out[key] = np.nanmean(arr, axis=0).tolist()
    max_len = max(len(hist['gates']) for hist in histories)
    arr_g = np.full((len(histories), max_len, n_modalities), np.nan, dtype=float)
    for row, hist in enumerate(histories):
        for col, gates in enumerate(hist['gates']):
            arr_g[row, col, :len(gates)] = gates
    out['gates'] = np.nanmean(arr_g, axis=0).tolist()
    return out

# ── Fusion on fixed OOF embeddings ────────────────────────────────────────
EMBED_DIM  = available['clinical'].shape[1]
n_subjects = ref_n
y_fusion   = ref_labels.copy()
all_embs   = torch.cat([available[nm] for nm in paired_names],dim=-1)
total_dim  = all_embs.shape[1]
print(f"  Fusion dim: {total_dim}")

mod_dims = {nm: available[nm].shape[1] for nm in paired_names}
fus_pf   = np.zeros(n_subjects, dtype=int)
fus_prf  = np.zeros(n_subjects, dtype=float)
fus_yf   = y_fusion.copy()
fusion_gate_means = []
fusion_histories  = []

print("\n  Training MoE Fusion (5-fold OOF)...")
for fold, (tri, vai) in enumerate(paired_cv_splits, start=1):
    set_seed(620 + fold)
    model_fold = MoEFusion(mod_dims, embed_dim=EMBED_DIM, temp=1.0).to(device)
    cw_f       = get_class_weights(y_fusion[tri],device)
    opt_f      = optim.Adam(model_fold.parameters(),lr=5e-4,weight_decay=1e-4)
    sc_f       = optim.lr_scheduler.CosineAnnealingLR(opt_f,T_max=200)
    crit_f     = nn.CrossEntropyLoss(weight=cw_f)
    es_f       = EarlyStopping(patience=20,mode='max')

    X_fus_tr_g = all_embs[tri].to(device)
    y_fus_tr_g = torch.LongTensor(y_fusion[tri]).to(device)
    X_fus_va_g = all_embs[vai].to(device)
    y_fus_va_g = torch.LongTensor(y_fusion[vai]).to(device)

    dl_f = DataLoader(TensorDataset(X_fus_tr_g,y_fus_tr_g),
                      batch_size=min(16,len(tri)),
                      sampler=make_weighted_sampler(y_fusion[tri]))

    fold_history = {'train_loss':[],'val_f1':[],'val_auc':[],'gates':[]}

    for ep in range(200):
        model_fold.train(); ep_loss=0; nb=0
        for xb,yb in dl_f:
            opt_f.zero_grad()
            logits,_ = model_fold(xb,return_gates=True)
            loss = crit_f(logits,yb)
            loss.backward(); opt_f.step()
            ep_loss+=loss.item(); nb+=1
        sc_f.step()
        model_fold.eval()
        with torch.no_grad():
            lv,gv = model_fold(X_fus_va_g,return_gates=True)
            pv    = lv.argmax(1).cpu().numpy()
            prv   = torch.softmax(lv,1)[:,1].cpu().numpy()
            yv_np = y_fus_va_g.cpu().numpy()
            f1v   = f1_score(yv_np,pv,zero_division=0)
            auv   = safe_auc(yv_np,prv)
        fold_history['train_loss'].append(ep_loss/max(nb,1))
        fold_history['val_f1'].append(f1v)
        fold_history['val_auc'].append(auv)
        fold_history['gates'].append(gv.mean(0).cpu().numpy())
        if (ep+1)%25==0:
            gw_str = '  '.join([f"{nm}={gv.mean(0).cpu().numpy()[i]:.3f}"
                                 for i,nm in enumerate(paired_names)])
            print(f"  [FusionCV{fold}] Ep {ep+1:3d}  f1={f1v:.4f}  "
                  f"auc={auv:.4f}  gates={gw_str}")
        if es_f.step(f1v,model_fold):
            print(f"  [FusionCV{fold}] Early stop ep {ep+1}  best_f1={es_f.best:.4f}")
            break
    es_f.restore(model_fold)
    fusion_histories.append(fold_history)

    model_fold.eval()
    with torch.no_grad():
        lvf,gvf = model_fold(X_fus_va_g,return_gates=True)
        fus_pf[vai]  = lvf.argmax(1).cpu().numpy()
        fus_prf[vai] = torch.softmax(lvf,1)[:,1].cpu().numpy()
        fold_auc     = safe_auc(y_fusion[vai], fus_prf[vai])
        fold_gate_means = gvf.mean(0).cpu().numpy()
        fusion_gate_means.append(fold_gate_means)
    gw_str = '  '.join([f"{nm}={fold_gate_means[i]:.3f}"
                         for i,nm in enumerate(paired_names)])
    print(f"  [FusionCV{fold}] Fold AUC={fold_auc:.4f}  gates={gw_str}")

history_f = aggregate_history(fusion_histories, len(paired_names))
mean_fusion_gates = np.mean(np.vstack(fusion_gate_means), axis=0)

# Fit a final compatibility model on all fixed OOF embeddings for save/export parity
print("\n  Fitting final fusion model on fixed OOF embeddings for save compatibility...")
set_seed(1042)
model_fusion = MoEFusion(mod_dims, embed_dim=EMBED_DIM, temp=1.0).to(device)
cw_f_all     = get_class_weights(y_fusion,device)
opt_f_all    = optim.Adam(model_fusion.parameters(),lr=5e-4,weight_decay=1e-4)
sched_f_all  = optim.lr_scheduler.ReduceLROnPlateau(opt_f_all,'max',factor=0.5,patience=6)
dl_f_all     = DataLoader(TensorDataset(all_embs.to(device),
                                        torch.LongTensor(y_fusion).to(device)),
                          batch_size=min(16,len(y_fusion)),
                          sampler=make_weighted_sampler(y_fusion))
train_nn(model_fusion, dl_f_all, all_embs.to(device),
         torch.LongTensor(y_fusion).to(device),
         criterion=nn.CrossEntropyLoss(weight=cw_f_all),
         optimizer=opt_f_all, scheduler=sched_f_all,
         max_epochs=80, patience=12, verbose_every=40, tag='FusionFinal')

# ── Evaluation ────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("EVALUATION")
print("="*65)

results = {}
results['fusion'] = {
    'acc':  float(accuracy_score(fus_yf,fus_pf)),
    'f1':   float(f1_score(fus_yf,fus_pf,zero_division=0)),
    'auc':  safe_auc(fus_yf,fus_prf),
    'prauc':float(average_precision_score(fus_yf,fus_prf)
                  if len(np.unique(fus_yf))>1 else 0.5),
    'gates':mean_fusion_gates,
    'pf':fus_pf,'prf':fus_prf,'yf':fus_yf
}

# Clinical — 5-fold CV on XGBoost (expert reference)
clin_cv_preds = np.zeros(len(y_clinical))
clin_cv_probs = np.zeros(len(y_clinical))
for tri,vai in cv5.split(X_clin_sel,y_clinical):
    sc_tmp = StandardScaler()
    Xtr_cv = sc_tmp.fit_transform(X_clin_sel[tri])
    Xva_cv = sc_tmp.transform(X_clin_sel[vai])
    m_tmp  = deepcopy(best_clin_model); m_tmp.fit(Xtr_cv,y_clinical[tri])
    clin_cv_preds[vai] = m_tmp.predict(Xva_cv)
    clin_cv_probs[vai] = m_tmp.predict_proba(Xva_cv)[:,1]
results['clinical'] = {
    'acc':float(accuracy_score(y_clinical,clin_cv_preds)),
    'f1': float(f1_score(y_clinical,clin_cv_preds,zero_division=0)),
    'auc':float(roc_auc_score(y_clinical,clin_cv_probs)),
    'yte':y_clinical,'ypr':clin_cv_probs
}

# Activity/HRV — aligned BioEncoder OOF reference on shared paired subjects
results['activity_hrv'] = {
    'acc':float(bio_aligned_oof_metrics['acc']),
    'f1': float(bio_aligned_oof_metrics['f1']),
    'auc':float(bio_aligned_oof_metrics['auc']),
    'auc_flipped': float(bio_aligned_oof_metrics['auc_flipped']),
    'yte':y_bio.copy(),'ypr':bio_aligned_oof_probs.copy()
}


def eval_lr(emb,lab):
    e = emb.numpy() if isinstance(emb,torch.Tensor) else np.array(emb)
    Xtr,Xte,ytr,yte = train_test_split(e,lab,test_size=0.2,
                                        stratify=lab,random_state=42)
    lr = LogisticRegression(max_iter=1000,class_weight='balanced',
                             random_state=42)
    lr.fit(Xtr,ytr); yp=lr.predict(Xte); ypr=lr.predict_proba(Xte)[:,1]
    return {'acc':float(accuracy_score(yte,yp)),
            'f1': float(f1_score(yte,yp,zero_division=0)),
            'auc':float(roc_auc_score(yte,ypr)
                        if len(np.unique(yte))>1 else 0.5),
            'yte':yte,'ypr':ypr}

if 'eeg' in available:
    results['eeg'] = eval_lr(available['eeg'],available_labels['eeg'])
if 'brain' in available:
    results['brain'] = eval_lr(available['brain'],available_labels['brain'])

paired_gate_reference = {
    'clinical': float(clinical_oof_metrics['auc']),
    'activity_hrv': float(bio_aligned_oof_metrics['auc']),
}
reference_order = sorted(paired_gate_reference,
                         key=lambda nm: paired_gate_reference[nm],
                         reverse=True)
gate_order = [nm for nm,_ in sorted(zip(paired_names,results['fusion']['gates']),
                                     key=lambda item: item[1],
                                     reverse=True)]

print(f"\n  {'Model':<22} {'N':>5} {'Acc':>8} {'F1':>8} {'AUC':>8}")
print("  " + "─"*55)
n_map = {'fusion':len(fus_yf),'clinical':len(y_clinical),
         'activity_hrv':len(y_bio),
         'eeg':len(available_labels.get('eeg',[])),
         'brain':len(available_labels.get('brain',[]))}
for nm in ['fusion','clinical','activity_hrv','eeg','brain']:
    if nm not in results: continue
    r=results[nm]; mk='★' if nm=='fusion' else ' '
    flag = '  ⚠' if r['auc']<0.55 else ''
    print(f"  {mk} {nm:<20} {n_map.get(nm,'?'):>5} "
          f"{r['acc']:>8.4f} {r['f1']:>8.4f} {r['auc']:>8.4f}{flag}")

print(f"\n  Gate weights:")
for nm,gw in zip(paired_names,results['fusion']['gates']):
    bar = "█"*int(gw*30)
    print(f"    {nm:<18}  {gw:.3f}  {bar}")

print("\n  Paired encoder OOF AUC reference:")
for nm in ['clinical','activity_hrv']:
    print(f"    {nm:<18}  AUC={paired_gate_reference[nm]:.4f}")

if gate_order != reference_order:
    print("\n  ⚠ Gate sanity warning: mean fusion gate ordering disagrees with paired encoder OOF AUC ordering.")
    print(f"    OOF AUC order : {reference_order}")
    print(f"    Gate order    : {gate_order}")
else:
    print("\n  ✅ Gate sanity check: mean fusion gate ordering matches paired encoder OOF AUC ordering.")

aucs_b = []
for _ in range(1000):
    idx_b = np.random.choice(len(fus_yf),len(fus_yf),replace=True)
    try: aucs_b.append(roc_auc_score(fus_yf[idx_b],fus_prf[idx_b]))
    except: pass
if len(aucs_b)>10:
    ci_lo,ci_hi = np.percentile(aucs_b,[2.5,97.5])
    print(f"\n  Fusion AUC: {results['fusion']['auc']:.4f}  "
          f"95% CI [{ci_lo:.3f},{ci_hi:.3f}]")

prec_a,rec_a,thr_a = precision_recall_curve(fus_yf,fus_prf)
f1_a     = 2*prec_a*rec_a/(prec_a+rec_a+1e-8)
best_thr = thr_a[np.argmax(f1_a[:-1])] if len(thr_a)>0 else 0.5
pf_opt   = (fus_prf>=best_thr).astype(int)
print(f"  Optimal threshold: {best_thr:.3f}  "
      f"→ F1={f1_score(fus_yf,pf_opt,zero_division=0):.4f}")

fusion_auc_target = 0.9444
fusion_auc_tolerance = 0.03
if results['fusion']['auc'] >= fusion_auc_target - fusion_auc_tolerance:
    print(f"  ✅ Fusion OOF AUC is within {fusion_auc_tolerance:.2f} of the 0.9444 target.")
else:
    print(f"  ⚠ Fusion OOF AUC {results['fusion']['auc']:.4f} is more than {fusion_auc_tolerance:.2f} below 0.9444.")


In [ ]:
# =============================================================================
# CELL 9 — EXPLAINABILITY LAYER
# =============================================================================
print("\n" + "="*65)
print("EXPLAINABILITY LAYER")
print("="*65)

def generate_nl_justification(shap_feats,shap_vals,pred_class,
                               confidence,modality):
    DSM5 = {
        'inattention':   ['rt','cpt','attention','error','commission',
                          'omission','theta','frontal','F3','F4','Fz'],
        'hyperactivity': ['activity','steps','movement','motor',
                          'act_mean','act_std'],
        'impulsivity':   ['rt_std','variability','reaction','beta'],
        'executive':     ['working','memory','switch','inhibit','stroop'],
        'sleep_arousal': ['sleep','hrv','rmssd','hr_','heart'],
    }
    top = [(shap_feats[i],shap_vals[i])
           for i in np.argsort(shap_vals)[::-1][:5]]
    triggered = set()
    for feat,val in top:
        fl = feat.lower()
        for crit,kws in DSM5.items():
            if any(k in fl for k in kws):
                triggered.add((crit,round(float(val),4),feat))
    label_str = "ADHD" if pred_class==1 else "Non-ADHD (likely HC)"
    lines = [
        f"Prediction: {label_str}  (confidence: {confidence*100:.1f}%)",
        f"Modality: {modality}", "",
        "Supporting evidence from top SHAP features:",
    ]
    for feat,val in top:
        lines.append(f"  • {feat:35s}  SHAP={val:+.4f}")
    if triggered:
        lines.append(""); lines.append("DSM-5 criterion mapping:")
        for crit,val,feat in sorted(triggered,key=lambda x:-x[1]):
            lines.append(f"  → {crit.replace('_',' ').title():<20} "
                         f"(driver: {feat}, |SHAP|={val:.4f})")
    if pred_class==1:
        lines.append(""); lines.append("Counterfactual direction:")
        for feat,val in top[:3]:
            d = "reduce" if val>0 else "increase"
            lines.append(f"  If {feat} were to {d}, confidence would decrease.")
    lines += ["","NOTE: AI-assisted decision support only.",
              "All outputs must be reviewed by a licensed clinician."]
    return "\n".join(lines)

adhd_idx  = np.where(y_clinical==1)[0][0] if sum(y_clinical==1)>0 else 0
x_example = X_clin_final[adhd_idx:adhd_idx+1]
pred_prob  = best_clin_model.predict_proba(x_example)[0][1]
pred_class = int(pred_prob>=0.5)
shap_ex    = explainer_clin.shap_values(x_example)
if isinstance(shap_ex,list): shap_ex = shap_ex[1]
shap_ex_vals = shap_ex.flatten()

print("\n--- CLINICAL EXPERT EXPLANATION ---")
print(generate_nl_justification(selected_feature_names,shap_ex_vals,
                                pred_class,pred_prob,
                                'Clinical (HYPERAKTIV CPT+actigraphy+HRV)'))

print("\n--- EEG EXPERT EXPLANATION ---")
print("Top EEG channels by SHAP importance (ADHD vs HC):")
for rank,ch_idx in enumerate(top_eeg_channels):
    ch_name = eeg_channels[ch_idx] if ch_idx<len(eeg_channels) else f"Ch{ch_idx}"
    print(f"  {rank+1}. {ch_name:<8}  importance={channel_importance_eeg[ch_idx]:.4f}")
print("\n  Elevated theta power at frontal channels (F3/F4/Fz) and")
print("  theta/beta ratio > 2.0 are primary ADHD EEG signatures.")

if 'fusion' in results:
    print("\n--- FUSION GATE WEIGHTS ---")
    for nm,gw in zip(paired_names,results['fusion']['gates']):
        flag = "← correctly downweighted" if gw<0.45 else "← correctly trusted"
        print(f"  {nm:<18}  weight={gw:.3f}  {flag}")

# Medication analysis
if has_med and medication_status is not None:
    med_mask = medication_status==1; no_med = medication_status==0
    if med_mask.sum()>2 and no_med.sum()>2:
        med_embs   = clinical_embeddings[med_mask].numpy()
        nomed_embs = clinical_embeddings[no_med].numpy()
        mm = med_embs.mean(0); nm2 = nomed_embs.mean(0)
        cos_sim = np.dot(mm,nm2)/(np.linalg.norm(mm)*np.linalg.norm(nm2)+1e-8)
        print(f"\n--- MEDICATION RESPONSE ---")
        print(f"  Cosine similarity (medicated vs unmedicated): {cos_sim:.4f}")
        print(f"  (Low similarity → medication shifts neural signature)")


In [ ]:
# =============================================================================
# CELL 10 — FIGURES
# =============================================================================
print("\n" + "="*65)
print("GENERATING FIGURES")
print("="*65)

MOD_COLORS = {'clinical':'#2196F3','activity_hrv':'#4CAF50',
              'eeg':'#FF9800','brain':'#E91E63','fusion':'#7B1FA2'}
DEFAULT_COLOR = '#9E9E9E'

display_order = ['fusion']+[n for n in
    ['clinical','activity_hrv','eeg','brain'] if n in results]

fig = plt.figure(figsize=(22,18),facecolor='white')
fig.suptitle('ADHD Explainable Multimodal AI — Complete Analysis',
             fontsize=16,fontweight='bold',y=0.98)
gs = gridspec.GridSpec(3,4,hspace=0.50,wspace=0.36,
                       left=0.06,right=0.96,top=0.93,bottom=0.05)

aucs_a  = [results[n]['auc'] for n in display_order]
cols_a  = [MOD_COLORS.get(n,DEFAULT_COLOR) for n in display_order]
ew_a    = [2.5 if n=='fusion' else 1.0 for n in display_order]
hatches = ['///' if v<0.55 else '' for v in aucs_a]

ax_a = fig.add_subplot(gs[0,0:2])
bars_a = ax_a.bar(display_order,aucs_a,color=cols_a,edgecolor='black',
                  linewidth=ew_a,hatch=hatches,zorder=3)
for bar,val,nm in zip(bars_a,aucs_a,display_order):
    c='red' if val<0.55 else 'black'; suf=' ⚠' if val<0.55 else ''
    ax_a.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
              f'{val:.3f}{suf}',ha='center',va='bottom',
              fontsize=9,fontweight='bold',color=c)
ax_a.axhline(0.5,ls='--',color='red',alpha=0.6,lw=1.2,label='Chance (0.5)')
ax_a.axhline(0.7,ls=':',color='gray',alpha=0.5,lw=1.0,label='Good (0.7)')
ax_a.set_ylim(0,1.18); ax_a.set_ylabel('ROC AUC')
ax_a.set_title('A. ROC AUC by modality  (⚠=below chance)',
               fontweight='bold',loc='left',fontsize=9)
ax_a.tick_params(axis='x',rotation=20); ax_a.grid(axis='y',alpha=0.3)
ax_a.legend(fontsize=8)

ax_b = fig.add_subplot(gs[0,2:4])
f1s_b = [results[n]['f1'] for n in display_order]
bars_b= ax_b.bar(display_order,f1s_b,color=cols_a,edgecolor='black',
                 linewidth=ew_a,hatch=hatches,zorder=3)
for bar,val in zip(bars_b,f1s_b):
    ax_b.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.01,
              f'{val:.3f}',ha='center',va='bottom',fontsize=10,fontweight='bold')
ax_b.set_ylim(0,1.12); ax_b.set_ylabel('F1 Score')
ax_b.set_title('B. F1 score by modality',fontweight='bold',loc='left')
ax_b.tick_params(axis='x',rotation=20); ax_b.grid(axis='y',alpha=0.3)

ax_c = fig.add_subplot(gs[1,0:2])
if len(history_f.get('gates',[]))>0:
    gh = np.array(history_f['gates']); n_gc = gh.shape[1]
    glabels = paired_names[:n_gc]
    for i in range(n_gc):
        nm = glabels[i] if i<len(glabels) else f'mod_{i}'
        ax_c.plot(gh[:,i],label=nm,color=MOD_COLORS.get(nm,DEFAULT_COLOR),lw=2)
        ax_c.annotate(f'{gh[-1,i]:.3f}',xy=(len(gh)-1,gh[-1,i]),
                      xytext=(len(gh)*0.75,gh[-1,i]+0.03),
                      fontsize=8,color=MOD_COLORS.get(nm,DEFAULT_COLOR))
    ax_c.axhline(1.0/n_gc,ls='--',color='gray',alpha=0.5,label='Uniform')
    ax_c.set_xlabel('Epoch'); ax_c.set_ylabel('Gate weight')
    ax_c.set_title('C. Gate weights — MoE downweights noisy modality',
                   fontweight='bold',loc='left',fontsize=9)
    ax_c.legend(fontsize=9); ax_c.grid(alpha=0.3)

ax_d = fig.add_subplot(gs[1,2:4])
if len(history_f.get('val_f1',[]))>0:
    ax_d2 = ax_d.twinx()
    ax_d.plot(history_f['train_loss'],color='#E57373',lw=1.5,label='Train loss')
    ax_d2.plot(history_f['val_f1'], color='#42A5F5',lw=2,label='Val F1')
    ax_d2.plot(history_f['val_auc'],color='#66BB6A',lw=2,ls='--',label='Val AUC')
    ax_d.set_xlabel('Epoch'); ax_d.set_ylabel('Loss',color='#E57373')
    ax_d2.set_ylabel('Score'); ax_d2.set_ylim(0,1.1)
    ax_d.set_title('D. Fusion training curves',fontweight='bold',loc='left')
    lines = ax_d.get_lines()+ax_d2.get_lines()
    ax_d.legend(lines,[l.get_label() for l in lines],fontsize=8)
    ax_d.grid(alpha=0.3)

ax_e = fig.add_subplot(gs[2,0])
cm_data = confusion_matrix(fus_yf,fus_pf)
sns.heatmap(cm_data,annot=True,fmt='d',cmap='Blues',ax=ax_e,
            xticklabels=['HC','ADHD'],yticklabels=['HC','ADHD'])
ax_e.set_xlabel('Predicted'); ax_e.set_ylabel('True')
ax_e.set_title(f'E. Fusion confusion matrix (N={len(fus_yf)})',
               fontweight='bold',loc='left')

ax_f = fig.add_subplot(gs[2,1])
roc_data = {}
if len(np.unique(fus_yf))>1:
    fpr,tpr,_ = roc_curve(fus_yf,fus_prf)
    roc_data['fusion']=(fpr,tpr,results['fusion']['auc'])
for nm in ['clinical','activity_hrv','eeg','brain']:
    if nm in results and 'ypr' in results[nm]:
        r=results[nm]
        if len(np.unique(r['yte']))>1:
            fp,tp,_ = roc_curve(r['yte'],r['ypr'])
            roc_data[nm]=(fp,tp,r['auc'])
for nm,(fpr,tpr,auc_v) in roc_data.items():
    lw=2.5 if nm=='fusion' else 1.5
    ls='-'  if nm=='fusion' else '--'
    al=1.0  if auc_v>=0.55 else 0.4
    ax_f.plot(fpr,tpr,color=MOD_COLORS.get(nm,DEFAULT_COLOR),lw=lw,
              ls=ls,alpha=al,
              label=f"{nm} ({auc_v:.3f})"+(" ⚠" if auc_v<0.55 else ""))
ax_f.plot([0,1],[0,1],'k--',alpha=0.4)
ax_f.set_xlabel('FPR'); ax_f.set_ylabel('TPR')
ax_f.set_title('F. ROC curves',fontweight='bold',loc='left')
ax_f.legend(fontsize=7,loc='lower right'); ax_f.grid(alpha=0.3)

ax_g = fig.add_subplot(gs[2,2])
if 'gates' in results['fusion']:
    ga  = results['fusion']['gates']
    gls = paired_names[:len(ga)]
    gcs = [MOD_COLORS.get(nm,DEFAULT_COLOR) for nm in gls]
    bars_g = ax_g.bar(gls,ga,color=gcs,edgecolor='black',linewidth=1)
    ax_g.axhline(1.0/len(ga),ls='--',color='red',alpha=0.5,label='Uniform')
    for bar,val,nm in zip(bars_g,ga,gls):
        auc_v = results.get(nm,{}).get('auc',0.5)
        ax_g.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.005,
                  f'{val:.3f}\n(AUC={auc_v:.2f})',
                  ha='center',va='bottom',fontsize=9,fontweight='bold')
    ax_g.set_ylabel('Gate weight')
    ax_g.set_title('G. MoE gate weights',fontweight='bold',loc='left')
    ax_g.legend(fontsize=8); ax_g.grid(axis='y',alpha=0.3)
    ax_g.set_ylim(0,max(ga)+0.18)

ax_h = fig.add_subplot(gs[2,3])
ax_h.axis('off')
trows = [['Model','N','F1','AUC','Note']]
n_map2 = {'fusion':len(fus_yf),'clinical':len(y_clinical),
          'activity_hrv':len(y_bio),
          'eeg':len(available_labels.get('eeg',[])),
          'brain':len(available_labels.get('brain',[]))}
for nm in display_order:
    if nm not in results: continue
    r=results[nm]; pfx='★' if nm=='fusion' else ''
    note='⚠ noise' if r['auc']<0.55 else 'gate↓' if r['auc']<0.65 else 'ok'
    trows.append([f"{pfx}{nm}",str(n_map2.get(nm,'?')),
                  f"{r['f1']:.3f}",f"{r['auc']:.3f}",note])
tbl = ax_h.table(cellText=trows[1:],colLabels=trows[0],
                 cellLoc='center',loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1,1.4)
for (row,col),cell in tbl.get_celld().items():
    if row==0:
        cell.set_facecolor('#1B3A6B')
        cell.set_text_props(color='white',fontweight='bold')
    elif row==1: cell.set_facecolor('#EDE7F6')
    elif row%2==0: cell.set_facecolor('#EEF4FB')
    if row>1:
        try:
            if float(trows[row][3])<0.55: cell.set_facecolor('#FFEBEE')
        except: pass
ax_h.set_title('H. Summary',fontweight='bold',loc='left')

plt.savefig('adhd_xai_results.png',dpi=150,bbox_inches='tight')
plt.show()
print("✅ Figure saved: adhd_xai_results.png")

# Bootstrap CI
aucs_b2 = []
for _ in range(1000):
    idx_b2 = np.random.choice(len(fus_yf),len(fus_yf),replace=True)
    try: aucs_b2.append(roc_auc_score(fus_yf[idx_b2],fus_prf[idx_b2]))
    except: pass
if len(aucs_b2)>10:
    ci_lo2,ci_hi2 = np.percentile(aucs_b2,[2.5,97.5])
    print(f"  Fusion AUC {results['fusion']['auc']:.4f}  "
          f"95% CI [{ci_lo2:.3f},{ci_hi2:.3f}]")
# ================================
# SAVE EACH SUBPLOT AS STANDALONE FIGURE
# ================================
import os

output_dir = '/kaggle/working/individual_plots'
os.makedirs(output_dir, exist_ok=True)

# --------------------------------------------------
# A. ROC AUC by modality
# --------------------------------------------------
fig_a, ax_a2 = plt.subplots(figsize=(8, 5))
bars_a2 = ax_a2.bar(display_order, aucs_a, color=cols_a, edgecolor='black',
                     linewidth=ew_a, hatch=hatches, zorder=3)
for bar, val in zip(bars_a2, aucs_a):
    c = 'red' if val < 0.55 else 'black'
    suf = ' ⚠' if val < 0.55 else ''
    ax_a2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
               f'{val:.3f}{suf}', ha='center', va='bottom',
               fontsize=10, fontweight='bold', color=c)
ax_a2.axhline(0.5, ls='--', color='red', alpha=0.6, lw=1.2, label='Chance (0.5)')
ax_a2.axhline(0.7, ls=':', color='gray', alpha=0.5, lw=1.0, label='Good (0.7)')
ax_a2.set_ylim(0, 1.18); ax_a2.set_ylabel('ROC AUC')
ax_a2.set_title('A. ROC AUC by Modality', fontweight='bold')
ax_a2.tick_params(axis='x', rotation=20); ax_a2.grid(axis='y', alpha=0.3)
ax_a2.legend(fontsize=9)
fig_a.savefig(os.path.join(output_dir, 'A_ROC_AUC.png'), format='png',
              dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig_a)
print("✅ Saved A_ROC_AUC.png")

# --------------------------------------------------
# B. F1 Score by modality
# --------------------------------------------------
fig_b, ax_b2 = plt.subplots(figsize=(8, 5))
bars_b2 = ax_b2.bar(display_order, f1s_b, color=cols_a, edgecolor='black',
                     linewidth=ew_a, hatch=hatches, zorder=3)
for bar, val in zip(bars_b2, f1s_b):
    ax_b2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
               f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax_b2.set_ylim(0, 1.12); ax_b2.set_ylabel('F1 Score')
ax_b2.set_title('B. F1 Score by Modality', fontweight='bold')
ax_b2.tick_params(axis='x', rotation=20); ax_b2.grid(axis='y', alpha=0.3)
fig_b.savefig(os.path.join(output_dir, 'B_F1_Score.png'), format='png',
              dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig_b)
print("✅ Saved B_F1_Score.png")

# --------------------------------------------------
# C. Gate weights over epochs
# --------------------------------------------------
if len(history_f.get('gates', [])) > 0:
    fig_c, ax_c2 = plt.subplots(figsize=(8, 5))
    gh = np.array(history_f['gates']); n_gc = gh.shape[1]
    glabels = paired_names[:n_gc]
    for i in range(n_gc):
        nm = glabels[i] if i < len(glabels) else f'mod_{i}'
        ax_c2.plot(gh[:, i], label=nm, color=MOD_COLORS.get(nm, DEFAULT_COLOR), lw=2)
        ax_c2.annotate(f'{gh[-1, i]:.3f}', xy=(len(gh)-1, gh[-1, i]),
                       fontsize=9, color=MOD_COLORS.get(nm, DEFAULT_COLOR))
    ax_c2.axhline(1.0/n_gc, ls='--', color='gray', alpha=0.5, label='Uniform')
    ax_c2.set_xlabel('Epoch'); ax_c2.set_ylabel('Gate Weight')
    ax_c2.set_title('C. Gate Weights — MoE Over Training', fontweight='bold')
    ax_c2.legend(fontsize=9); ax_c2.grid(alpha=0.3)
    fig_c.savefig(os.path.join(output_dir, 'C_Gate_Weights.png'), format='png',
                  dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig_c)
    print("✅ Saved C_Gate_Weights.png")

# --------------------------------------------------
# D. Training curves
# --------------------------------------------------
if len(history_f.get('val_f1', [])) > 0:
    fig_d, ax_d2 = plt.subplots(figsize=(8, 5))
    ax_d2_twin = ax_d2.twinx()
    ax_d2.plot(history_f['train_loss'], color='#E57373', lw=1.5, label='Train loss')
    ax_d2_twin.plot(history_f['val_f1'], color='#42A5F5', lw=2, label='Val F1')
    ax_d2_twin.plot(history_f['val_auc'], color='#66BB6A', lw=2, ls='--', label='Val AUC')
    ax_d2.set_xlabel('Epoch'); ax_d2.set_ylabel('Loss', color='#E57373')
    ax_d2_twin.set_ylabel('Score'); ax_d2_twin.set_ylim(0, 1.1)
    ax_d2.set_title('D. Fusion Training Curves', fontweight='bold')
    lines = ax_d2.get_lines() + ax_d2_twin.get_lines()
    ax_d2.legend(lines, [l.get_label() for l in lines], fontsize=9)
    ax_d2.grid(alpha=0.3)
    fig_d.savefig(os.path.join(output_dir, 'D_Training_Curves.png'), format='png',
                  dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig_d)
    print("✅ Saved D_Training_Curves.png")

# --------------------------------------------------
# E. Confusion matrix
# --------------------------------------------------
fig_e, ax_e2 = plt.subplots(figsize=(6, 5))
cm_data = confusion_matrix(fus_yf, fus_pf)
sns.heatmap(cm_data, annot=True, fmt='d', cmap='Blues', ax=ax_e2,
            xticklabels=['HC', 'ADHD'], yticklabels=['HC', 'ADHD'])
ax_e2.set_xlabel('Predicted'); ax_e2.set_ylabel('True')
ax_e2.set_title(f'E. Fusion Confusion Matrix (N={len(fus_yf)})', fontweight='bold')
fig_e.savefig(os.path.join(output_dir, 'E_Confusion_Matrix.png'), format='png',
              dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig_e)
print("✅ Saved E_Confusion_Matrix.png")

# --------------------------------------------------
# F. ROC curves
# --------------------------------------------------
fig_f, ax_f2 = plt.subplots(figsize=(7, 6))
for nm, (fpr, tpr, auc_v) in roc_data.items():
    lw = 2.5 if nm == 'fusion' else 1.5
    ls = '-' if nm == 'fusion' else '--'
    al = 1.0 if auc_v >= 0.55 else 0.4
    ax_f2.plot(fpr, tpr, color=MOD_COLORS.get(nm, DEFAULT_COLOR), lw=lw,
               ls=ls, alpha=al,
               label=f"{nm} ({auc_v:.3f})" + (" ⚠" if auc_v < 0.55 else ""))
ax_f2.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax_f2.set_xlabel('FPR'); ax_f2.set_ylabel('TPR')
ax_f2.set_title('F. ROC Curves', fontweight='bold')
ax_f2.legend(fontsize=9, loc='lower right'); ax_f2.grid(alpha=0.3)
fig_f.savefig(os.path.join(output_dir, 'F_ROC_Curves.png'), format='png',
              dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig_f)
print("✅ Saved F_ROC_Curves.png")

# --------------------------------------------------
# G. MoE gate weights bar chart
# --------------------------------------------------
if 'gates' in results['fusion']:
    fig_g, ax_g2 = plt.subplots(figsize=(7, 5))
    ga = results['fusion']['gates']
    gls = paired_names[:len(ga)]
    gcs = [MOD_COLORS.get(nm, DEFAULT_COLOR) for nm in gls]
    bars_g2 = ax_g2.bar(gls, ga, color=gcs, edgecolor='black', linewidth=1)
    ax_g2.axhline(1.0/len(ga), ls='--', color='red', alpha=0.5, label='Uniform')
    for bar, val, nm in zip(bars_g2, ga, gls):
        auc_v = results.get(nm, {}).get('auc', 0.5)
        ax_g2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                   f'{val:.3f}\n(AUC={auc_v:.2f})',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax_g2.set_ylabel('Gate Weight')
    ax_g2.set_title('G. MoE Gate Weights', fontweight='bold')
    ax_g2.legend(fontsize=9); ax_g2.grid(axis='y', alpha=0.3)
    ax_g2.set_ylim(0, max(ga) + 0.18)
    fig_g.savefig(os.path.join(output_dir, 'G_MoE_Weights.png'), format='png',
                  dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig_g)
    print("✅ Saved G_MoE_Weights.png")

# --------------------------------------------------
# H. Summary table
# --------------------------------------------------
fig_h, ax_h2 = plt.subplots(figsize=(8, 4))
ax_h2.axis('off')
trows = [['Model', 'N', 'F1', 'AUC', 'Note']]
n_map2 = {'fusion': len(fus_yf), 'clinical': len(y_clinical),
           'activity_hrv': len(y_bio),
           'eeg': len(available_labels.get('eeg', [])),
           'brain': len(available_labels.get('brain', []))}
for nm in display_order:
    if nm not in results: continue
    r = results[nm]; pfx = '★' if nm == 'fusion' else ''
    note = '⚠ noise' if r['auc'] < 0.55 else 'gate↓' if r['auc'] < 0.65 else 'ok'
    trows.append([f"{pfx}{nm}", str(n_map2.get(nm, '?')),
                  f"{r['f1']:.3f}", f"{r['auc']:.3f}", note])
tbl = ax_h2.table(cellText=trows[1:], colLabels=trows[0],
                   cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.6)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1B3A6B')
        cell.set_text_props(color='white', fontweight='bold')
    elif row == 1: cell.set_facecolor('#EDE7F6')
    elif row % 2 == 0: cell.set_facecolor('#EEF4FB')
ax_h2.set_title('H. Summary Table', fontweight='bold')
fig_h.savefig(os.path.join(output_dir, 'H_Summary_Table.png'), format='png',
              dpi=150, bbox_inches='tight', facecolor='white')
plt.close(fig_h)
print("✅ Saved H_Summary_Table.png")

# --------------------------------------------------
# List all saved files
# --------------------------------------------------
print(f"\n📁 All individual plots saved to: {output_dir}")
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    print(f"   📊 {f}  ({os.path.getsize(fpath)/1024:.1f} KB)")
print("\n👉 Go to: Output tab (right sidebar) → individual_plots/ → Download each")

In [ ]:
# =============================================================================
# CELL 11 — SAVE ALL OUTPUTS
# =============================================================================
save_out = {
    'timestamp': datetime.now().isoformat(),
    'datasets_used': {
        'clinical_hrv': 'kaggle.com/datasets/kishore00afk/hyperaktiv',
        'eeg':          'kaggle.com/datasets/danizo/eeg-dataset-for-adhd',
        'brain':        'kaggle.com/datasets/kishore00afk/adhd-200-preprocessed',
    },
    'fixes_applied': [
        'Text branch removed — invalid ADHD-vs-ADHD labels',
        'WiDS competition data replaced with open ADHD-200',
        'HYPERAKTIV clinical+activity/HRV = real paired data',
        'SHAP added for all expert models',
        'activity_hrv gate calibrated on aligned BioEncoder OOF embeddings',
        'Gate sanity check compares MoE weights against paired encoder OOF AUC',
    ],
    'results': {
        k: {m: round(float(v),4)
            for m,v in r.items()
            if m not in ('gates','pf','prf','yf','yte','ypr')}
        for k,r in results.items()
    },
}

if 'gates' in results['fusion']:
    save_out['fusion_gate_weights'] = {
        nm: round(float(gw),4)
        for nm,gw in zip(paired_names[:len(results['fusion']['gates'])],
                         results['fusion']['gates'])
    }

with open('adhd_xai_results.json','w') as f:
    json.dump(save_out,f,indent=2)
print("✅ Saved: adhd_xai_results.json")

for var,fname in [('model_clinical','model_clinical.pth'),
                  ('model_eeg',     'model_eeg.pth'),
                  ('model_fusion',  'model_fusion.pth'),
                  ('model_brain',   'model_brain.pth'),
                  ('model_bio',     'model_bio.pth')]:
    m = globals().get(var,None)
    if m is not None:
        torch.save(m.state_dict(),fname)
        print(f"✅ Saved: {fname}")

print("\n" + "="*65)
print("COMPLETE")
print("="*65)
for nm in display_order:
    if nm not in results: continue
    r=results[nm]; mk='★' if nm=='fusion' else ' '
    flag = '  ⚠ below chance' if r['auc']<0.55 else ''
    print(f"  {mk} {nm:<22}  F1={r['f1']:.4f}  AUC={r['auc']:.4f}{flag}")
print("\n  Key XAI finding:")
for nm,gw in zip(paired_names,results['fusion']['gates']):
    auc_v = results.get(nm,{}).get('auc',0.5)
    d = 'correctly downweighted' if gw<0.45 else 'correctly trusted'
    print(f"    Gate {nm:<18} = {gw:.3f}  AUC={auc_v:.3f}  [{d}]")


In [ ]:
# =============================================================================
# CELL 12 - EXPORT PREPROCESSING ARTIFACTS
# =============================================================================
print("\n" + "="*65)
print("EXPORT PREPROCESSING ARTIFACTS")
print("="*65)

EXPORT_DIR = '/kaggle/working/preprocessing'
os.makedirs(EXPORT_DIR, exist_ok=True)

def serialize_scaler(sc):
    return {
        'mean': sc.mean_.tolist(),
        'scale': sc.scale_.tolist(),
        'var': sc.var_.tolist(),
        'n_features_in': int(sc.n_features_in_),
        'n_samples_seen': int(sc.n_samples_seen_),
    }

clinical_bundle = {
    'bundle_type': 'clinical_selected_features',
    'source_notebook': 'adha-new-method (2).ipynb',
    'input_key': 'clinical_selected',
    'selected_feature_names': list(selected_feature_names),
    'feature_names': list(selected_feature_names),
    'scaler': serialize_scaler(sc_c),
    'metadata': {
        'subject_count': int(len(y_clinical)),
        'selected_feature_count': int(len(selected_feature_names)),
        'random_state': 42,
        'generated_at': datetime.now().isoformat(),
    },
}

activity_hrv_bundle = {
    'bundle_type': 'activity_hrv_features',
    'source_notebook': 'adha-new-method (2).ipynb',
    'input_key': 'activity_hrv_features',
    'feature_names': list(activity_hrv_export_feature_names),
    'scaler': serialize_scaler(activity_hrv_export_scaler),
    'metadata': {
        'subject_count': int(len(y_bio)),
        'feature_count': int(len(activity_hrv_export_feature_names)),
        'random_state': 42,
        'generated_at': datetime.now().isoformat(),
        'alignment': activity_hrv_export_metadata['alignment'],
        'final_model': activity_hrv_export_metadata['final_model'],
        'benchmark': activity_hrv_export_metadata['benchmark'],
    },
}

with open(os.path.join(EXPORT_DIR, 'clinical_bundle.json'), 'w', encoding='utf-8') as f:
    json.dump(clinical_bundle, f, indent=2)
with open(os.path.join(EXPORT_DIR, 'activity_hrv_bundle.json'), 'w', encoding='utf-8') as f:
    json.dump(activity_hrv_bundle, f, indent=2)

pd.DataFrame([{feat: 0 for feat in selected_feature_names}]).to_csv(
    os.path.join(EXPORT_DIR, 'clinical_selected_template.csv'), index=False)
pd.DataFrame([{feat: 0 for feat in selected_feature_names}]).to_json(
    os.path.join(EXPORT_DIR, 'clinical_selected_template.json'), orient='records', indent=2)
pd.DataFrame([{feat: 0 for feat in activity_hrv_export_feature_names}]).to_csv(
    os.path.join(EXPORT_DIR, 'activity_hrv_template.csv'), index=False)
pd.DataFrame([{feat: 0 for feat in activity_hrv_export_feature_names}]).to_json(
    os.path.join(EXPORT_DIR, 'activity_hrv_template.json'), orient='records', indent=2)

manifest = {
    'generated_at': datetime.now().isoformat(),
    'export_dir': EXPORT_DIR,
    'files': {
        'clinical_bundle': 'clinical_bundle.json',
        'activity_hrv_bundle': 'activity_hrv_bundle.json',
        'clinical_selected_template_csv': 'clinical_selected_template.csv',
        'clinical_selected_template_json': 'clinical_selected_template.json',
        'activity_hrv_template_csv': 'activity_hrv_template.csv',
        'activity_hrv_template_json': 'activity_hrv_template.json',
    },
}
with open(os.path.join(EXPORT_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print(f"Saved clinical bundle: {os.path.join(EXPORT_DIR, 'clinical_bundle.json')}")
print(f"Saved activity_hrv bundle: {os.path.join(EXPORT_DIR, 'activity_hrv_bundle.json')}")
print(f"Saved templates and manifest to: {EXPORT_DIR}")
